<a href="https://colab.research.google.com/github/trinovita03/Pneumonia-Object-Detection/blob/main/Faster%20R-CNN%20dengan%20Efficient%20Net%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip show timm

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip install pydicom opencv-python pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
base_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia"
train_dicom_path = f"{base_path}/stage_2_train_images"
label_csv_path = f"{base_path}/stage_2_train_labels.csv"
output_jpg_path = f"{base_path}/train_jpg"


In [ ]:
#MENAMPILKAN DATA CSV
import pandas as pd
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)
df.head()


In [ ]:
import pandas as pd
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)
df.info()


In [ ]:
import matplotlib.pyplot as plt

# Data distribusi kelas
labels = ["Normal", "Pneumonia"]
counts = [20672, 6012]

# Membuat grafik batang
plt.figure(figsize=(6, 4))
plt.bar(labels, counts)
plt.title("Distribusi Data Kelas Normal dan Pneumonia")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Data")

# Menampilkan jumlah di atas batang
for i, value in enumerate(counts):
    plt.text(i, value, str(value), ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
import os
import pydicom
import matplotlib.pyplot as plt

dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"

# Ambil 1 file DICOM pertama
sample_file = os.listdir(dicom_dir)[0]
sample_path = os.path.join(dicom_dir, sample_file)

dcm = pydicom.dcmread(sample_path)

plt.imshow(dcm.pixel_array, cmap='gray')
plt.title(f"Citra DICOM (Patient ID: {dcm.PatientID})")
plt.axis('off')
plt.show()


In [ ]:
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import os
import numpy as np

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"

df = pd.read_csv(labels_path)

normal_id = df[df['Target'] == 0]['patientId'].iloc[0]
pneumonia_id = df[df['Target'] == 1]['patientId'].iloc[0]

def load_dicom(path):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min())
    return img

img_normal = load_dicom(os.path.join(dicom_dir, normal_id + ".dcm"))
img_pneumonia = load_dicom(os.path.join(dicom_dir, pneumonia_id + ".dcm"))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_normal, cmap="gray")
plt.title("NORMAL (Target = 0)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_pneumonia, cmap="gray")
plt.title("PNEUMONIA (Target = 1)")
plt.axis("off")

# Keterangan format DICOM
plt.suptitle("Citra rontgen dada dalam format DICOM", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
#MENAMPILKAN BOUNDING BOX DATA ASLI FORMAT DICOM
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import os
import matplotlib.patches as patches

# Path
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"

# Load label
df = pd.read_csv(labels_path)

# Ambil 1 contoh normal & pneumonia
normal_id = df[df['Target'] == 0]['patientId'].iloc[0]
pneumonia_id = df[df['Target'] == 1]['patientId'].iloc[0]

# Load DICOM RAW
dcm_normal = pydicom.dcmread(os.path.join(dicom_dir, normal_id + ".dcm"))
dcm_pneumonia = pydicom.dcmread(os.path.join(dicom_dir, pneumonia_id + ".dcm"))

img_normal = dcm_normal.pixel_array
img_pneumonia = dcm_pneumonia.pixel_array

# Ambil bounding box pneumonia
bbox_df = df[df['patientId'] == pneumonia_id]

# Plot
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# ---- NORMAL ----
axs[0].imshow(img_normal, cmap='gray')
axs[0].set_title("NORMAL (RAW DICOM, tanpa bounding box)")
axs[0].axis('off')

# ---- PNEUMONIA ----
axs[1].imshow(img_pneumonia, cmap='gray')

for _, row in bbox_df.iterrows():
    x, y, w, h = row['x'], row['y'], row['width'], row['height']
    rect = patches.Rectangle(
        (x, y), w, h,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    axs[1].add_patch(rect)

axs[1].set_title("PNEUMONIA (RAW DICOM + Bounding Box)")
axs[1].axis('off')

plt.suptitle(
    "Visualisasi data asli dataset RSNA Pneumonia Detection Challenge",
    fontsize=12
)
plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import pydicom
import numpy as np
import time
from tqdm import tqdm

# Folder asal dan tujuan
train_dicom_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"
output_jpg_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

# Buat folder output jika belum ada
os.makedirs(output_jpg_path, exist_ok=True)

start_time = time.time()

# Ambil semua file DICOM
dicom_files = [f for f in os.listdir(train_dicom_path) if f.endswith(".dcm")]
print(f"Total file DICOM ditemukan: {len(dicom_files)}")

# Loop konversi semua file DICOM
for f in tqdm(dicom_files, desc="Mengonversi DICOM ke JPG"):
    dicom_path = os.path.join(train_dicom_path, f)
    jpg_path = os.path.join(output_jpg_path, f.replace(".dcm", ".jpg"))

    try:
        # Baca file DICOM
        dcm = pydicom.dcmread(dicom_path)
        img = dcm.pixel_array

        # Normalisasi ke rentang 0–255
        img = cv2.convertScaleAbs(img, alpha=(255.0 / np.max(img)))

        # Simpan sebagai JPG
        cv2.imwrite(jpg_path, img)

    except Exception as e:
        print(f"Gagal konversi {f}: {e}")

end_time = time.time()
processing_time = end_time - start_time

print(f"Waktu pemrosesan: {processing_time:.4f} detik")

print("✅ Konversi selesai! Semua file disimpan dalam folder train_jpg/")



In [ ]:
!ls "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images" | wc -l

In [ ]:
#jumlah data yang dikonversi
import os
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"
jpg_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

print("Jumlah DICOM :", len(os.listdir(dicom_dir)))
print("Jumlah JPG   :", len(os.listdir(jpg_dir)))



In [ ]:
#MENGECEK RESOLUSI DATA
import os, cv2, pydicom
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"
jpg_dir   = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

dicom_file = os.listdir(dicom_dir)[0]
jpg_file   = os.listdir(jpg_dir)[0]

# Load DICOM
dcm = pydicom.dcmread(os.path.join(dicom_dir, dicom_file))
dicom_shape = dcm.pixel_array.shape

# Load JPG
img = cv2.imread(os.path.join(jpg_dir, jpg_file), cv2.IMREAD_GRAYSCALE)
jpg_shape = img.shape

print("Resolusi DICOM :", dicom_shape)
print("Resolusi JPG   :", jpg_shape)


In [ ]:
import random
import matplotlib.pyplot as plt
import cv2

# Ambil 1 file JPG acak dari folder hasil
jpg_files = os.listdir(output_jpg_path)
sample_jpg = random.choice(jpg_files)

# Baca dan tampilkan
img = cv2.imread(os.path.join(output_jpg_path, sample_jpg))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Contoh hasil konversi: {sample_jpg}")
plt.axis('off')
plt.show()


In [ ]:
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import os

# Path
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
img_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

# Load label
df = pd.read_csv(labels_path)

# Ambil 1 contoh
normal_id = df[df['Target'] == 0]['patientId'].iloc[0]
pneumonia_id = df[df['Target'] == 1]['patientId'].iloc[0]

# Load gambar JPG
img_normal = cv2.imread(os.path.join(img_dir, normal_id + ".jpg"),
                         cv2.IMREAD_GRAYSCALE)
img_pneumonia = cv2.imread(os.path.join(img_dir, pneumonia_id + ".jpg"),
                            cv2.IMREAD_GRAYSCALE)

# Plot sampingan
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_normal, cmap="gray")
plt.title("NORMAL (Target = 0)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_pneumonia, cmap="gray")
plt.title("PNEUMONIA (Target = 1)")
plt.axis("off")

# Keterangan format
plt.suptitle("Citra rontgen dada dalam format JPG", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
#CLEANING DATA

In [ ]:
import pandas as pd
import os

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"

df = pd.read_csv(labels_path)

# Jumlah file citra
dicom_files = [f.replace(".dcm", "") for f in os.listdir(dicom_dir) if f.endswith(".dcm")]

print("Jumlah file DICOM:", len(dicom_files))
print("Jumlah patientId unik di CSV:", df['patientId'].nunique())


In [ ]:
dicom_set = set(dicom_files)
csv_set = set(df['patientId'].unique())

missing_in_csv = dicom_set - csv_set

print("Jumlah citra tanpa label:", len(missing_in_csv))
print("patientId tanpa label:", list(missing_in_csv)[:5])


In [ ]:
import os

# Folder JPG asli
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

# File tanpa label
file_sampah = "576f3fdc-8864-4709-b72f-a7b33fee8435 (1).jpg"

path_file = os.path.join(input_dir, file_sampah)

if os.path.exists(path_file):
    os.remove(path_file)
    print(f"🗑️ Berhasil menghapus: {file_sampah}")
else:
    print("File tidak ditemukan di folder JPG")


In [ ]:
df_clean = df[df['patientId'].isin(dicom_set)]
print("Jumlah data setelah cleaning:", df_clean['patientId'].nunique())

In [ ]:
# Ambil hanya patientId yang punya pneumonia (Target = 1)
df_pneumonia = df[df['Target'] == 1]

pneumonia_ids = set(df_pneumonia['patientId'])
dicom_ids = set(dicom_files)

# Gambar yang TIDAK punya bbox
no_bbox = dicom_ids - pneumonia_ids

print("Jumlah gambar TANPA bbox:", len(no_bbox))
print("Contoh:", list(no_bbox)[:10])

In [ ]:
print("Jumlah pneumonia:", df[df['Target'] == 1]['patientId'].nunique())
print("Jumlah normal:", df[df['Target'] == 0]['patientId'].nunique())

In [ ]:
import os

image_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"

image_files = [f.replace(".jpg", "")
               for f in os.listdir(image_dir)
               if f.endswith(".jpg")]

print("Jumlah file gambar:", len(image_files))
print("Jumlah patientId unik di CSV:", df['patientId'].nunique())


In [ ]:
import pandas as pd
import os

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
dicom_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_images"

df = pd.read_csv(labels_path)

print("Jumlah data awal:", len(df))


In [ ]:
unique_images = df['patientId'].nunique()
print("Jumlah gambar unik berdasarkan CSV:", unique_images)


In [ ]:
normal_count = df[df['Target'] == 0]['patientId'].nunique()
pneumonia_count = df[df['Target'] == 1]['patientId'].nunique()

normal_count, pneumonia_count


In [ ]:
import pandas as pd

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"

df = pd.read_csv(labels_path)


In [ ]:
import os
import shutil
import pandas as pd

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
img_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"
filtered_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"

os.makedirs(filtered_dir, exist_ok=True)

df = pd.read_csv(labels_path)

# Ambil hanya pneumonia (yang punya bbox)
pneumonia_ids = set(df[df['Target'] == 1]['patientId'])

count = 0

for file in os.listdir(img_dir):
    if not file.endswith(".jpg"):
        continue

    pid = file.replace(".jpg","")

    if pid in pneumonia_ids:
        src = os.path.join(img_dir, file)
        dst = os.path.join(filtered_dir, file)
        shutil.copy(src, dst)
        count += 1

print("Total gambar yang dicopy:", count)

In [ ]:
import cv2
import os
import pandas as pd
import shutil
from concurrent.futures import ThreadPoolExecutor

input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"
output_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"
label_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"

os.makedirs(output_dir, exist_ok=True)

# baca label
labels = pd.read_csv(label_path)

# ambil patientId yang memiliki target 1
target1_ids = set(labels[labels["Target"] == 1]["patientId"])

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def process_image(file_name):
    if not file_name.endswith(".jpg"):
        return

    patient_id = file_name.replace(".jpg", "")
    img_path = os.path.join(input_dir, file_name)
    output_path = os.path.join(output_dir, file_name)

    if os.path.exists(output_path):
        return

    # jika target 1 → CLAHE
    if patient_id in target1_ids:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("Gagal membaca:", file_name)
            return

        img_clahe = clahe.apply(img)
        cv2.imwrite(output_path, img_clahe)

    # jika target 0 → copy saja
    else:
        shutil.copy(img_path, output_path)

jpg_files = os.listdir(input_dir)

with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_image, jpg_files)

print("CLAHE hanya diterapkan pada Target = 1 selesai diproses")

In [ ]:
import pandas as pd
import random

# 1. Baca semua label asli
df_asli = pd.read_csv('/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv')

# 2. Ambil semua ID yang sudah dipakai di TRAINING (Pneumonia saja)
train_ids_used = set(train_df['patientId'].unique())
val_pneu_ids = set(val_df['patientId'].unique())
test_pneu_ids = set(test_df['patientId'].unique())

# Gabungkan semua ID yang sudah "terpakai" untuk pneumonia
all_used_ids = train_ids_used | val_pneu_ids | test_pneu_ids

# 3. Ambil semua ID yang statusnya NORMAL (Target 0)
# dan pastikan TIDAK ADA di dalam list pneumonia di atas
all_normal_pool = df_asli[df_asli['Target'] == 0]['patientId'].unique()
available_normal = [pid for pid in all_normal_pool if pid not in all_used_ids]

print(f"Total ID Normal tersedia: {len(available_normal)}")

# 4. Ambil 601 untuk Val dan 602 untuk Test (Random tapi Terkunci/Seed)
random.seed(42)
random.shuffle(available_normal)

val_normal_ids = available_normal[:601]
test_normal_ids = available_normal[601:1203] # 601 + 602 = 1203

# Simpan ke Dataframe
df_val_normal = pd.DataFrame({'patientId': val_normal_ids, 'Target': 0})
df_test_normal = pd.DataFrame({'patientId': test_normal_ids, 'Target': 0})

print(f"✅ Berhasil memisahkan:")
print(f"- Val Normal : {len(df_val_normal)} ID")
print(f"- Test Normal: {len(df_test_normal)} ID")
print(f"Apakah ada ID yang sama? {set(val_normal_ids).intersection(set(test_normal_ids))}") # Harus set() kosong

In [ ]:
import cv2
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 1. DEFINISIKAN PATH (Penting!) ---
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_jpg"
output_dir_val = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/val_normal_clahe"
output_dir_test = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/test_normal_clahe"

# Buat foldernya jika belum ada
os.makedirs(output_dir_val, exist_ok=True)
os.makedirs(output_dir_test, exist_ok=True)

# --- 2. DEFINISIKAN FUNGSI CLAHE ---
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def process_safe_normal(pid, target_folder):
    output_path = os.path.join(target_folder, f"{pid}.jpg")

    # Lewati jika file sudah ada
    if os.path.exists(output_path):
        return

    # Cari gambar asli (pastikan ekstensinya .jpg sesuai folder train_images kamu)
    img_path = os.path.join(input_dir, f"{pid}.jpg")

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        img_clahe = clahe.apply(img)
        cv2.imwrite(output_path, img_clahe)
    else:
        # Opsional: print jika ada file yang benar-benar tidak ketemu
        # print(f"File {pid}.jpg tidak ditemukan di {input_dir}")
        pass

# --- 3. JALANKAN PROSES ---

# Proses Val Normal (601 Data)
print(f"⏳ Memproses {len(val_normal_ids)} CLAHE Val Normal ke: {output_dir_val}")
for pid in tqdm(val_normal_ids):
    process_safe_normal(pid, output_dir_val)

# Proses Test Normal (602 Data)
print(f"⏳ Memproses {len(test_normal_ids)} CLAHE Test Normal ke: {output_dir_test}")
for pid in tqdm(test_normal_ids):
    process_safe_normal(pid, output_dir_test)

print("\n✅ Semua proses CLAHE Normal selesai!")
print(f"Jumlah file di Val Normal: {len(os.listdir(output_dir_val))}")
print(f"Jumlah file di Test Normal: {len(os.listdir(output_dir_test))}")

In [ ]:
import os
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"
output_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"

print("Jumlah asli:", len(os.listdir(input_dir)))
print("Jumlah CLAHE:", len(os.listdir(output_dir)))

In [ ]:
import pandas as pd
import os

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"

df = pd.read_csv(labels_path)

# Ambil hanya pneumonia
df_pneumonia = df[df['Target'] == 1]
csv_ids = set(df_pneumonia['patientId'])

# Ambil ID dari folder
img_ids = set([
    f.replace(".jpg","").replace("_hflip","").replace("_vflip","")
    for f in os.listdir(input_dir)
    if f.endswith(".jpg")
])

print("Jumlah ID pneumonia di CSV:", len(csv_ids))
print("Jumlah ID di folder:", len(img_ids))

In [ ]:
#Cek gambar TANPA anotasi
no_bbox = img_ids - csv_ids

print("Gambar tanpa bbox:", len(no_bbox))
print("Contoh:", list(no_bbox)[:10])

In [ ]:
missing_img = csv_ids - img_ids

print("Data CSV tanpa gambar:", len(missing_img))
print("Contoh:", list(missing_img)[:10])

In [ ]:
#MENAMPILKAN 1 GAMBAR CLAHE UNTUK DILIHAT GRIDENYA
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Baca gambar grayscale
img = cv2.imread("/content/sample_xray.jpg", cv2.IMREAD_GRAYSCALE)

# Terapkan CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_img = clahe.apply(img)


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

folder = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"

files = [f for f in os.listdir(folder) if f.endswith(".jpg")]

img_path = os.path.join(folder, files[0])  # ambil 1 gambar

img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise ValueError("Gambar tidak ditemukan atau path salah")

h, w = img.shape
tile_h = h // 8
tile_w = w // 8

img_grid = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

for i in range(1, 8):
    cv2.line(img_grid, (0, i * tile_h), (w, i * tile_h), (0,255,0), 1)
    cv2.line(img_grid, (i * tile_w, 0), (i * tile_w, h), (0,255,0), 1)

plt.figure(figsize=(6,6))
plt.imshow(img_grid)
plt.title("Grid CLAHE 8x8")
plt.axis("off")
plt.show()

In [ ]:
tile = img[0:tile_h, 0:tile_w]

plt.figure(figsize=(5,3))
plt.hist(tile.ravel(), bins=256)
plt.title("Histogram Tile (0,0)")
plt.show()


In [ ]:
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
img_clahe = clahe.apply(img)

tile_clahe = img_clahe[0:tile_h, 0:tile_w]

plt.figure(figsize=(10,3))

plt.subplot(1,2,1)
plt.hist(tile.ravel(), bins=256)
plt.title("Sebelum CLAHE")

plt.subplot(1,2,2)
plt.hist(tile_clahe.ravel(), bins=256)
plt.title("Sesudah CLAHE")

plt.show()


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe.apply(img)

tile_clahe = img_clahe[0:tile_h, 0:tile_w]

plt.figure(figsize=(10,3))

plt.subplot(1,2,1)
plt.hist(tile.ravel(), bins=256)
plt.title("Sebelum CLAHE")

plt.subplot(1,2,2)
plt.hist(tile_clahe.ravel(), bins=256)
plt.title("Sesudah CLAHE")

plt.show()


In [ ]:
import os
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"
output_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"

print("Jumlah JPG asli:", len(os.listdir(input_dir)))
print("Jumlah JPG CLAHE:", len(os.listdir(output_dir)))

In [ ]:
#VISUALISASI PERBANDINGAN BEFORE AFTER CLAHE 2 KELAS TERSEBUT
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd

# Folder JPG asli dan hasil CLAHE
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"
clahe_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"

# CSV label
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)

# Ambil 1 contoh pneumonia dan 1 contoh normal
pid_pneumonia = df[df['Target'] == 1]['patientId'].unique()[5]

# Baca gambar
img_pneumonia_orig = cv2.imread(os.path.join(input_dir, pid_pneumonia + ".jpg"), cv2.IMREAD_GRAYSCALE)
img_pneumonia_clahe = cv2.imread(os.path.join(clahe_dir, pid_pneumonia + ".jpg"), cv2.IMREAD_GRAYSCALE)

img_normal_orig = cv2.imread(os.path.join(input_dir, pid_normal + ".jpg"), cv2.IMREAD_GRAYSCALE)
img_normal_clahe = cv2.imread(os.path.join(clahe_dir, pid_normal + ".jpg"), cv2.IMREAD_GRAYSCALE)

# Visualisasi
plt.figure(figsize=(12, 8))

# Pneumonia
plt.subplot(2, 2, 1)
plt.imshow(img_pneumonia_orig, cmap='gray')
plt.title("Pneumonia - Original")
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(img_pneumonia_clahe, cmap='gray')
plt.title("Pneumonia - CLAHE")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd

# Path
input_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_pneumonia_only"
clahe_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"

df = pd.read_csv(labels_path)

# Ambil 10 patient pneumonia
patient_ids = df[df['Target'] == 1]['patientId'].unique()[10:20]

plt.figure(figsize=(20, 20))

for i, pid in enumerate(patient_ids):
    img_orig = cv2.imread(os.path.join(input_dir, pid + ".jpg"), cv2.IMREAD_GRAYSCALE)
    img_clahe = cv2.imread(os.path.join(clahe_dir, pid + ".jpg"), cv2.IMREAD_GRAYSCALE)

    # Original
    plt.subplot(10, 2, 2*i + 1)
    plt.imshow(img_orig, cmap='gray')
    plt.title(f"Pneumonia - Original ({pid})", fontsize=8)
    plt.axis('off')

    # CLAHE
    plt.subplot(10, 2, 2*i + 2)
    plt.imshow(img_clahe, cmap='gray')
    plt.title(f"Pneumonia - CLAHE ({pid})", fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#SPLIT DATA

In [ ]:
import pandas as pd

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)


In [ ]:
df = df[df['Target'] == 1]
trainval_df = df.copy()

In [ ]:
import pandas as pd

# Pastikan hanya data pneumonia (punya bounding box)
df_pneumonia = df[df['Target'] == 1]

# Hitung jumlah bounding box per patientId
bbox_per_patient = (
    df_pneumonia
    .groupby('patientId')
    .size()
    .reset_index(name='num_boxes')
)

# Hitung distribusi jumlah box
distribution = (
    bbox_per_patient
    .groupby('num_boxes')
    .size()
    .reset_index(name='num_images')
)

print(distribution)


In [ ]:
#MENAMPILKAN 4 JENIS BOUNDING BOX
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import os

# === PATH DATA ===
ANNOTATION_PATH = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
IMAGE_DIR = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"

# === LOAD ANNOTATION ===
df = pd.read_csv(ANNOTATION_PATH)
df = df[df["Target"] == 1]

# Hitung jumlah bounding box per citra
bbox_count = df.groupby("patientId").size().reset_index(name="num_boxes")

# Ambil contoh citra dengan 1–4 bounding box
examples = {}
for n in [1, 2, 3, 4]:
    sample = bbox_count[bbox_count["num_boxes"] == n]
    if not sample.empty:
        examples[n] = sample.iloc[0]["patientId"]

# === VISUALISASI ===
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for idx, (num_boxes, patient_id) in enumerate(examples.items()):
    image_path = os.path.join(IMAGE_DIR, patient_id + ".jpg")
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    ax = axes[idx]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"{num_boxes} Bounding Box")
    ax.axis("off")

    boxes = df[df["patientId"] == patient_id]
    for _, row in boxes.iterrows():
        rect = patches.Rectangle(
            (row["x"], row["y"]),
            row["width"],
            row["height"],
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )
        ax.add_patch(rect)

plt.tight_layout()
plt.show()


In [ ]:
#DATAFRAME SPLIT DATA
from sklearn.model_selection import train_test_split
import pandas as pd

labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)

df_pneumonia = df[df['Target'] == 1]
bbox_count_df = (
    df_pneumonia
    .groupby('patientId')
    .size()
    .reset_index(name='num_boxes')
)

patient_df = bbox_count_df.copy()
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.2,
    stratify=patient_df['num_boxes'],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    stratify=temp_patients['num_boxes'],
    random_state=42
)

train_df = df_pneumonia[df_pneumonia['patientId'].isin(train_patients['patientId'])]
val_df   = df_pneumonia[df_pneumonia['patientId'].isin(val_patients['patientId'])]
test_df  = df_pneumonia[df_pneumonia['patientId'].isin(test_patients['patientId'])]

print(train_df['patientId'].nunique(),
      val_df['patientId'].nunique(),
      test_df['patientId'].nunique())

In [ ]:
#SPLIT DATA
import shutil
import os

clahe_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/train_clahe"
base_out = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split"

train_out = os.path.join(base_out, "train")
val_out   = os.path.join(base_out, "val")
test_out  = os.path.join(base_out, "test")

os.makedirs(train_out, exist_ok=True)
os.makedirs(val_out, exist_ok=True)
os.makedirs(test_out, exist_ok=True)

train_ids = set(train_df['patientId'])
val_ids   = set(val_df['patientId'])
test_ids  = set(test_df['patientId'])

for file in os.listdir(clahe_dir):
    if not file.endswith(".jpg"):
        continue

    pid = file.replace(".jpg", "")

    src = os.path.join(clahe_dir, file)

    if pid in train_ids:
        shutil.copy(src, os.path.join(train_out, file))
    elif pid in val_ids:
        shutil.copy(src, os.path.join(val_out, file))
    elif pid in test_ids:
        shutil.copy(src, os.path.join(test_out, file))


In [ ]:
import pandas as pd

def check_bbox_distribution(df_subset, subset_name):
    bbox_count = (
        df_subset
        .groupby('patientId')
        .size()
        .reset_index(name='num_boxes')
        .groupby('num_boxes')
        .size()
        .reset_index(name='num_patients')
    )

    print(f"\nDistribusi bounding box pada {subset_name}:")
    print(bbox_count)

# Cek distribusi di masing-masing subset
check_bbox_distribution(train_df, "Training Set")
check_bbox_distribution(val_df, "Validation Set")
check_bbox_distribution(test_df, "Test Set")


In [ ]:
#AUGMENTASI

In [ ]:
# AUGMENTASI DATA (SETELAH SPLIT, HANYA TRAIN)
import cv2
import os

# Folder TRAIN hasil split CLAHE
train_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split/train"

# Folder output augmentasi
aug_dir = "/content/train_aug"
os.makedirs(aug_dir, exist_ok=True)

for file_name in os.listdir(train_dir):
    if not file_name.endswith(".jpg"):
        continue

    img_path = os.path.join(train_dir, file_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    base_name = file_name.replace(".jpg", "")

    # Horizontal flip
    img_hflip = cv2.flip(img, 1)

    # Vertical flip
    img_vflip = cv2.flip(img, 0)

    # Simpan hasil augmentasi
    cv2.imwrite(os.path.join(aug_dir, f"{base_name}_hflip.jpg"), img_hflip)
    cv2.imwrite(os.path.join(aug_dir, f"{base_name}_vflip.jpg"), img_vflip)

print("Augmentasi data TRAIN selesai (horizontal & vertical flipping).")


In [ ]:
#FLIP ANOTASI BOUNDING BOX DI FILE CSVNYA
import torch
from torch.utils.data import Dataset
import cv2
import os

class RSNADataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.files = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        # ===== DETEKSI AUGMENTASI =====
        is_hflip = "_hflip" in file
        is_vflip = "_vflip" in file

        pid = file.replace(".jpg","").replace("_hflip","").replace("_vflip","")

        # ===== LOAD IMAGE =====
        img_path = os.path.join(self.img_dir, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Gagal load image: {file}")

        h, w = img.shape

        img = torch.tensor(img).float().unsqueeze(0) / 255.0

        # ===== AMBIL BBOX DARI CSV =====
        rows = self.df[self.df.patientId == pid]

        # ===== HANDLE KALAU TIDAK ADA BBOX =====
        if len(rows) == 0:
            target = {
                "boxes": torch.zeros((0,4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
            return img, target

        boxes = rows[['x','y','width','height']].values

        # ===== CONVERT KE (x1, y1, x2, y2) =====
        boxes[:,2] += boxes[:,0]
        boxes[:,3] += boxes[:,1]

        # ===== FLIP BBOX =====
        if is_hflip:
            x_min = boxes[:,0].copy()
            x_max = boxes[:,2].copy()
            boxes[:,0] = w - x_max
            boxes[:,2] = w - x_min

        if is_vflip:
            y_min = boxes[:,1].copy()
            y_max = boxes[:,3].copy()
            boxes[:,1] = h - y_max
            boxes[:,3] = h - y_min

        # ===== CLAMP (BIAR TIDAK KELUAR GAMBAR) =====
        boxes[:,0] = boxes[:,0].clip(0, w)
        boxes[:,2] = boxes[:,2].clip(0, w)
        boxes[:,1] = boxes[:,1].clip(0, h)
        boxes[:,3] = boxes[:,3].clip(0, h)

        # ===== VALIDASI BBOX =====
        valid_boxes = []
        for box in boxes:
            x1, y1, x2, y2 = box

            if x2 <= x1 or y2 <= y1:
                continue

            valid_boxes.append([x1, y1, x2, y2])

        # ===== KALAU SEMUA BBOX INVALID =====
        if len(valid_boxes) == 0:
            print(f"WARNING: bbox invalid semua di {pid}")

            target = {
                "boxes": torch.zeros((0,4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
            return img, target

        boxes = torch.tensor(valid_boxes).float()

        target = {
            "boxes": boxes,
            "labels": torch.ones(len(boxes), dtype=torch.int64)
        }

        return img, target


# ===== COLLATE FUNCTION =====
def collate_fn(batch):
    images = []
    targets = []

    for img, tgt in batch:
        images.append(img)
        targets.append(tgt)

    return images, targets

In [ ]:
#Menambahkan citra ori ke folder yang sama dengan augmentasinya
import os
import shutil

train_clahe_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split/train"
train_aug_dir   = "/content/train_aug"

os.makedirs(train_aug_dir, exist_ok=True)

for file in os.listdir(train_clahe_dir):
    if file.endswith(".jpg"):
        shutil.copy(
            os.path.join(train_clahe_dir, file),
            os.path.join(train_aug_dir, file)
        )

print("✅ Citra TRAIN asli (CLAHE) berhasil disalin ke folder train_aug")


In [ ]:
import os

aug_dir = "/content/train_aug"

jumlah_aug = len([f for f in os.listdir(aug_dir) if f.endswith(".jpg")])

print("Jumlah citra hasil augmentasi:", jumlah_aug)

In [ ]:
#Jumlah Data Yang Akan di Training Setelah Augmentasi Flip vertikal dan Horizontal
import os

aug_dir = "/content/train_aug"

ori = len([
    f for f in os.listdir(aug_dir)
    if f.endswith(".jpg") and "_hflip" not in f and "_vflip" not in f
])

hflip = len([f for f in os.listdir(aug_dir) if "_hflip" in f])
vflip = len([f for f in os.listdir(aug_dir) if "_vflip" in f])

print("Jumlah citra asli (ORI)       :", ori)
print("Jumlah citra horizontal flip :", hflip)
print("Jumlah citra vertical flip   :", vflip)
print("Jumlah seluruh citra TRAIN   :", ori + hflip + vflip)

In [ ]:
#MENAMPILKAN CONTOH AUGMENTASI YANG DILAKUKAN
import os
import cv2
import matplotlib.pyplot as plt

aug_dir = "/content/train_aug"

# Ambil 3 file ORI (tanpa suffix)
ori_files = [
    f for f in os.listdir(aug_dir)
    if f.endswith(".jpg") and "_hflip" not in f and "_vflip" not in f
][:3]

fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for i, ori_file in enumerate(ori_files):
    base = ori_file.replace(".jpg", "")

    ori = cv2.imread(os.path.join(aug_dir, ori_file), cv2.IMREAD_GRAYSCALE)
    h   = cv2.imread(os.path.join(aug_dir, f"{base}_hflip.jpg"), cv2.IMREAD_GRAYSCALE)
    v   = cv2.imread(os.path.join(aug_dir, f"{base}_vflip.jpg"), cv2.IMREAD_GRAYSCALE)

    axes[i, 0].imshow(ori, cmap="gray")
    axes[i, 0].set_title("Asli (CLAHE)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(h, cmap="gray")
    axes[i, 1].set_title("Horizontal Flip")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(v, cmap="gray")
    axes[i, 2].set_title("Vertical Flip")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#Menampilkan Hasil Augmentasi Flip Data CSV
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import os

# ====== CONFIG ======
img_dir = "/content/train_aug"       # Folder augmentasi
csv_file = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"  # CSV asli berisi bounding box
patient_ids_to_show = None  # List patientId yang ingin ditampilkan, None = semua

# Load CSV
df = pd.read_csv(csv_file)

# Ambil file gambar augmentasi
files = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

# Fungsi menampilkan gambar dengan bounding box
def show_img_with_boxes(img_path, boxes):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    for box in boxes:
        x_min, y_min, x_max, y_max = box
        cv2.rectangle(img, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (255,0,0), 2)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

# Loop untuk menampilkan semua gambar
for file in files:
    pid = file.replace(".jpg","").replace("_hflip","").replace("_vflip","")

    if patient_ids_to_show is not None and pid not in patient_ids_to_show:
        continue  # skip jika tidak termasuk list

    img_path = os.path.join(img_dir, file)

    # Ambil bounding box asli dari CSV
    rows = df[df.patientId == pid]
    boxes = rows[['x','y','width','height']].values
    boxes[:,2] += boxes[:,0]  # x_max
    boxes[:,3] += boxes[:,1]  # y_max

    h, w = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE).shape

    # Flip bounding box jika diperlukan
    if "_hflip" in file:
        x_min = boxes[:,0].copy()
        x_max = boxes[:,2].copy()
        boxes[:,0] = w - x_max
        boxes[:,2] = w - x_min

    if "_vflip" in file:
        y_min = boxes[:,1].copy()
        y_max = boxes[:,3].copy()
        boxes[:,1] = h - y_max
        boxes[:,3] = h - y_min

    # Tampilkan gambar dengan bounding box
    print(f"Patient ID: {pid} | File: {file}")
    show_img_with_boxes(img_path, boxes)


In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import os

# ====== CONFIG ======
img_dir = "/content/train_aug"       # Folder augmentasi
csv_file = "/content/drive/MyDrive/RSNA_Pneumonia/stage_2_train_labels.csv"  # CSV asli
patient_ids_to_show = None  # None = ambil otomatis beberapa patient
num_examples = 3  # jumlah patientId yang ditampilkan

# Load CSV
df = pd.read_csv(csv_file)

# Ambil file gambar augmentasi
files = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

# Buat dict: patientId -> list file (ori, hflip, vflip)
from collections import defaultdict
pid_files = defaultdict(dict)

for file in files:
    pid = file.replace(".jpg","").replace("_hflip","").replace("_vflip","")
    if "_hflip" in file:
        pid_files[pid]['hflip'] = file
    elif "_vflip" in file:
        pid_files[pid]['vflip'] = file
    else:
        pid_files[pid]['ori'] = file

# Ambil patientId yang akan ditampilkan
all_pids = list(pid_files.keys()) if patient_ids_to_show is None else patient_ids_to_show
all_pids = all_pids[:num_examples]  # batasi jumlah

# Fungsi untuk load bounding box dan flip jika perlu
def get_boxes(file_name, pid):
    rows = df[df.patientId == pid]
    boxes = rows[['x','y','width','height']].values
    boxes[:,2] += boxes[:,0]  # x_max
    boxes[:,3] += boxes[:,1]  # y_max

    img_path = os.path.join(img_dir, file_name)
    h, w = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE).shape

    # Flip bbox jika perlu
    if "_hflip" in file_name:
        x_min = boxes[:,0].copy()
        x_max = boxes[:,2].copy()
        boxes[:,0] = w - x_max
        boxes[:,2] = w - x_min

    if "_vflip" in file_name:
        y_min = boxes[:,1].copy()
        y_max = boxes[:,3].copy()
        boxes[:,1] = h - y_max
        boxes[:,3] = h - y_min

    return img_path, boxes

# Fungsi untuk menampilkan 3 jenis gambar secara horizontal
def show_patient_images(pid):
    fig, axes = plt.subplots(1, 3, figsize=(15,5))

    for i, kind in enumerate(['ori','hflip','vflip']):
        if kind not in pid_files[pid]:
            axes[i].axis('off')
            continue
        file_name = pid_files[pid][kind]
        img_path, boxes = get_boxes(file_name, pid)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        for box in boxes:
            x_min, y_min, x_max, y_max = box
            cv2.rectangle(img, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (255,0,0), 2)

        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(kind)

    plt.suptitle(f"Patient ID: {pid}")
    plt.show()

# Tampilkan contoh
for pid in all_pids:
    show_patient_images(pid)


In [ ]:
#Diagram Sebelum dan Sesudah Augmentasi bagian Training set Per Bounding Box
import pandas as pd

# =========================
# Hitung jumlah bbox per patient (TRAIN)
# =========================
bbox_per_patient = (
    train_df
    .groupby("patientId")
    .size()
)

# Distribusi: bbox_count -> jumlah gambar
dist_before = bbox_per_patient.value_counts().sort_index()

dist_before

In [ ]:
# Karena augmentasi: asli + hflip + vflip
dist_after = dist_before * 3

dist_after

In [ ]:
dist_df = pd.DataFrame({
    "Bounding Box per Gambar": dist_before.index,
    "Sebelum Augmentasi": dist_before.values,
    "Sesudah Augmentasi": dist_after.values
})

dist_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(dist_df))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, dist_df["Sebelum Augmentasi"], width, label="Sebelum Augmentasi")
plt.bar(x + width/2, dist_df["Sesudah Augmentasi"], width, label="Sesudah Augmentasi")

plt.xticks(x, dist_df["Bounding Box per Gambar"])
plt.xlabel("Jumlah Bounding Box per Gambar")
plt.ylabel("Jumlah Gambar")
plt.title("Distribusi Bounding Box Training Set\nSebelum dan Sesudah Augmentasi")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#PEMBUATAN MODEL

In [ ]:
# ==============================================================================
# CELL 1: IMPORTS
# ==============================================================================

from collections import OrderedDict
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# Torchvision detection components
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.ops import FeaturePyramidNetwork
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool

In [ ]:
# ==============================================================================
# CELL 2: EFFICIENTNET BACKBONE WRAPPER
# ==============================================================================

class EfficientNetBackbone(nn.Module):
    """
    EfficientNetV2 backbone wrapper for object detection.

    Uses timm's `features_only=True` mode to extract intermediate feature maps
    at multiple scales, which are essential for FPN.

    EfficientNetV2-S Architecture Output Channels:
    - Stage 0: 24 channels  (1/2 resolution)
    - Stage 1: 48 channels  (1/4 resolution)
    - Stage 2: 64 channels  (1/8 resolution)
    - Stage 3: 128 channels (1/16 resolution)
    - Stage 4: 160 channels (1/32 resolution)
    - Stage 5: 256 channels (1/32 resolution)

    We use stages 2-5 for FPN (1/8 to 1/32 resolution) as is standard practice.
    Earlier stages have too high resolution and would be memory-intensive.
    """

    def __init__(
        self,
        model_name: str = 'tf_efficientnet_b0',
        pretrained: bool = True,
        out_indices: Tuple[int, ...] = (2, 3, 4, 5),
        freeze_bn: bool = False
    ):
        """
        Initialize EfficientNet backbone.

        Args:
            model_name: timm model name (tf_efficientnet_b0, tf_efficientnetv2_m, etc.)
            pretrained: Use ImageNet pretrained weights
            out_indices: Which feature stages to output (0-indexed)
            freeze_bn: Freeze BatchNorm layers (useful for small batch sizes)
        """
        super().__init__()

        # Create timm model with feature extraction mode
        self.body = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=out_indices
        )

        # Get output channel dimensions for each stage
        # This is critical for connecting to FPN
        self.out_channels_list = self.body.feature_info.channels()
        self.out_indices = out_indices

        print(f"EfficientNetV2 Backbone initialized:")
        print(f"  Model: {model_name}")
        print(f"  Output stages: {out_indices}")
        print(f"  Output channels: {self.out_channels_list}")

        # Optionally freeze BatchNorm
        # Recommended for batch_size < 4 to prevent BN stats corruption
        if freeze_bn:
            self._freeze_bn()

    def _freeze_bn(self):
        """Freeze all BatchNorm layers."""
        for module in self.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                module.eval()
                for param in module.parameters():
                    param.requires_grad = False

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass returning multi-scale features.

        Args:
            x: Input tensor of shape (B, 3, H, W)

        Returns:
            OrderedDict mapping feature names to tensors
            Keys are '0', '1', '2', '3' corresponding to out_indices
        """
        # Get features from all specified stages
        features = self.body(x)

        # Convert to OrderedDict with string keys (required by FPN)
        out = OrderedDict()
        for idx, feat in enumerate(features):
            out[str(idx)] = feat

        return out

    @property
    def out_channels(self) -> int:
        """Return the number of output channels after FPN (unified)."""
        # FPN unifies all channels to the same dimension
        # We'll set this in the full model
        return 256


In [ ]:
# ==============================================================================
# CELL 3: CUSTOM FPN WITH EFFICIENTNET
# ==============================================================================

class EfficientNetWithFPN(nn.Module):
    """
    EfficientNetV2 backbone combined with Feature Pyramid Network.

    WHY FPN?
    ========
    1. Multi-scale detection: Pneumonia opacities vary greatly in size
    2. Combines low-level (edges, textures) and high-level (semantic) features
    3. Standard in all modern object detectors (Faster R-CNN, RetinaNet, YOLO)

    FPN Architecture:
    - Takes multi-scale features from backbone (C2, C3, C4, C5)
    - Creates pyramid features (P2, P3, P4, P5) with same channel dimension
    - Adds top-down pathway with lateral connections
    - Optionally adds P6 via max pooling for very large objects
    """

    def __init__(
        self,
        backbone_name: str = 'tf_efficientnet_b0',
        pretrained: bool = True,
        fpn_out_channels: int = 256,
        freeze_bn: bool = False,
        extra_blocks: bool = False,
    ):
        """
        Initialize backbone + FPN.

        Args:
            backbone_name: timm model name
            pretrained: Use pretrained weights
            fpn_out_channels: Number of channels in FPN output (256 is standard)
            freeze_bn: Freeze BatchNorm layers
            extra_blocks: Add extra max-pool level (P6)
        """
        super().__init__()

        # Create backbone
        self.backbone = EfficientNetBackbone(
            model_name=backbone_name,
            pretrained=pretrained,
            out_indices=(1, 2, 3, 4),  # Use stages with 1/8 to 1/32 resolution
            freeze_bn=freeze_bn
        )

        # Get input channels for FPN from backbone
        in_channels_list = self.backbone.out_channels_list

        # Create FPN
        # extra_blocks adds P6 level for detecting large objects
        extra = LastLevelMaxPool() if extra_blocks else None

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=fpn_out_channels,
            extra_blocks=None
        )

        # Store output channels for detection head
        self.out_channels = fpn_out_channels

        print(f"\nFPN initialized:")
        print(f"  Input channels: {in_channels_list}")
        print(f"  Output channels: {fpn_out_channels}")
        print(f"  Extra P6 level: {extra_blocks}")

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass through backbone and FPN.

        Args:
            x: Input tensor (B, 3, H, W)

        Returns:
            Dictionary of FPN features {'0': P2, '1': P3, '2': P4, '3': P5, 'pool': P6}
        """
        # Extract multi-scale features from backbone
        backbone_features = self.backbone(x)

        # Pass through FPN
        fpn_features = self.fpn(backbone_features)

        return fpn_features



In [ ]:
def create_anchor_generator() -> AnchorGenerator:
    """
    Anchor generator adapted from pneumonia X-ray analysis using K-Means++.
    Anchor scales are based on clustered bounding box sizes in the dataset.
    """

    # 3 anchor sizes (derived from dataset statistics)
    # mapped to FPN levels (P3, P4, P5)
    anchor_sizes = (
        (32,),    # small pneumonia regions
        (64,),   # medium regions
        (128,),
        (256,),   # large regions
    )

    # Aspect ratios based on article
    aspect_ratios = (
        (0.5, 1.0, 2.0),
    ) * 4

    anchor_generator = AnchorGenerator(
        sizes=anchor_sizes,
        aspect_ratios=aspect_ratios
    )

    return anchor_generator


In [ ]:
# ==============================================================================
# CELL 5: ROI POOLER CONFIGURATION
# ==============================================================================

def get_roi_pooler_config() -> Dict:
    """
    Configuration for RoI (Region of Interest) pooling.

    RoI Align is used instead of RoI Pool for better accuracy.
    It uses bilinear interpolation to avoid quantization artifacts.
    """
    return {
        'featmap_names': ['0', '1', '2', '3'],  # FPN levels to use
        'output_size': 7,  # Output spatial size (7x7 is standard)
        'sampling_ratio': 2  # Sampling points for RoI Align
    }



In [ ]:
# ==============================================================================
# CELL 6: MAIN PNEUMONIA DETECTOR CLASS
# ==============================================================================

class PneumoniaDetector(nn.Module):
    """
    Complete Pneumonia Detection Model.

    Architecture:
    1. EfficientNetV2-S backbone (timm)
    2. Feature Pyramid Network
    3. Region Proposal Network (RPN)
    4. RoI Align + Detection Head (Faster R-CNN)

    This is a two-stage detector:
    - Stage 1 (RPN): Proposes candidate regions
    - Stage 2 (Detection): Classifies and refines boxes

    Configuration Notes:
    - num_classes=2: Background (0) + Pneumonia (1)
    - Image mean/std: ImageNet normalization (handled in dataset)
    - NMS threshold: 0.5 for RPN, 0.3 for detection (will use WBF post-hoc)
    """

    def __init__(
        self,
        backbone_name: str = 'tf_efficientnet_b0',
        num_classes: int = 2,  # Background + Pneumonia
        pretrained_backbone: bool = True,
        # RPN settings
        rpn_pre_nms_top_n_train: int = 2000,
        rpn_pre_nms_top_n_test: int = 1000,
        rpn_post_nms_top_n_train: int = 2000,
        rpn_post_nms_top_n_test: int = 1000,
        rpn_nms_thresh: float = 0.7,
        rpn_fg_iou_thresh: float = 0.7,
        rpn_bg_iou_thresh: float = 0.3,
        # Detection settings
        box_score_thresh: float = 0.05,  # Low threshold - filter with WBF later
        box_nms_thresh: float = 0.5,
        box_detections_per_img: int = 100,
        box_fg_iou_thresh: float = 0.5,
        box_bg_iou_thresh: float = 0.5,
        # Training settings
        freeze_bn: bool = True,  # Freeze BN for small batch sizes
        trainable_backbone_layers: int = 5,  # Fine-tune all layers
    ):
        super().__init__()

        self.num_classes = num_classes
        self.backbone_name = backbone_name

        print("=" * 60)
        print("INITIALIZING PNEUMONIA DETECTOR")
        print("=" * 60)

        # 1. Create backbone with FPN
        backbone_fpn = EfficientNetWithFPN(
            backbone_name=backbone_name,
            pretrained=pretrained_backbone,
            fpn_out_channels=256,
            freeze_bn=freeze_bn,
            extra_blocks=True  # Add P6 level
        )

        # 2. Create anchor generator
        anchor_generator = create_anchor_generator()

        # 3. Create RPN head
        # RPN predicts objectness and box deltas for each anchor
        rpn_head = RPNHead(
            in_channels=backbone_fpn.out_channels,
            num_anchors=anchor_generator.num_anchors_per_location()[0]
        )

        # 4. Create complete Faster R-CNN model
        self.model = FasterRCNN(
            backbone=backbone_fpn,
            num_classes=num_classes,
            # RPN parameters
            rpn_anchor_generator=anchor_generator,
            rpn_head=rpn_head,
            rpn_pre_nms_top_n_train=rpn_pre_nms_top_n_train,
            rpn_pre_nms_top_n_test=rpn_pre_nms_top_n_test,
            rpn_post_nms_top_n_train=rpn_post_nms_top_n_train,
            rpn_post_nms_top_n_test=rpn_post_nms_top_n_test,
            rpn_nms_thresh=rpn_nms_thresh,
            rpn_fg_iou_thresh=rpn_fg_iou_thresh,
            rpn_bg_iou_thresh=rpn_bg_iou_thresh,
            rpn_batch_size_per_image=256,
            rpn_positive_fraction=0.5,
            # Box parameters
            box_score_thresh=box_score_thresh,
            box_nms_thresh=box_nms_thresh,
            box_detections_per_img=box_detections_per_img,
            box_fg_iou_thresh=box_fg_iou_thresh,
            box_bg_iou_thresh=box_bg_iou_thresh,
            box_batch_size_per_image=512,
            box_positive_fraction=0.25,
        )

        print(f"\nFaster R-CNN initialized:")
        print(f"  Num classes: {num_classes}")
        print(f"  RPN NMS thresh: {rpn_nms_thresh}")
        print(f"  Box score thresh: {box_score_thresh}")
        print(f"  Box NMS thresh: {box_nms_thresh}")
        print("=" * 60)

    def forward(
        self,
        images: List[torch.Tensor],
        targets: Optional[List[Dict[str, torch.Tensor]]] = None
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass.

        In training mode:
            Returns loss dictionary {'loss_classifier', 'loss_box_reg',
                                     'loss_objectness', 'loss_rpn_box_reg'}

        In inference mode:
            Returns list of detection dictionaries per image
            [{'boxes': Tensor, 'labels': Tensor, 'scores': Tensor}, ...]

        Args:
            images: List of tensors, each (3, H, W)
            targets: List of target dicts (only needed for training)

        Returns:
            Losses (training) or detections (inference)
        """
        return self.model(images, targets)

    def get_trainable_parameters(
        self,
        lr_backbone: float = 1e-5,
        lr_fpn: float = 1e-4,
        lr_head: float = 1e-3
    ) -> List[Dict]:
        """
        Get parameter groups with different learning rates.

        WHY DIFFERENT LEARNING RATES?
        =============================
        - Backbone: Pretrained, needs small LR to preserve learned features
        - FPN: New layers, can use moderate LR
        - Detection head: New layers, highest LR for fast convergence

        This is called "discriminative learning rates" and is critical
        for fine-tuning pretrained models.
        """
        # Separate parameters by component
        backbone_params = []
        fpn_params = []
        head_params = []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            if 'backbone.backbone' in name:
                backbone_params.append(param)
            elif 'backbone.fpn' in name:
                fpn_params.append(param)
            else:
                head_params.append(param)

        param_groups = [
            {'params': backbone_params, 'lr': lr_backbone, 'name': 'backbone'},
            {'params': fpn_params, 'lr': lr_fpn, 'name': 'fpn'},
            {'params': head_params, 'lr': lr_head, 'name': 'head'},
        ]

        print(f"\nParameter groups:")
        print(f"  Backbone: {len(backbone_params)} params, lr={lr_backbone}")
        print(f"  FPN: {len(fpn_params)} params, lr={lr_fpn}")
        print(f"  Head: {len(head_params)} params, lr={lr_head}")

        return param_groups

    def freeze_backbone(self, freeze: bool = True):
        """Freeze/unfreeze the backbone for transfer learning."""
        for param in self.model.backbone.backbone.parameters():
            param.requires_grad = not freeze
        print(f"Backbone {'frozen' if freeze else 'unfrozen'}")

    def freeze_bn(self):
        """Set all BatchNorm layers to eval mode."""
        for module in self.model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                module.eval()

In [ ]:
# ==============================================================================
# CELL 7: MODEL FACTORY (EFFICIENTNET-B0)
# ==============================================================================

def create_efficientnet_b0_fasterrcnn(
    num_classes: int = 2,
    pretrained: bool = True,
    **kwargs
) -> PneumoniaDetector:
    """
    Create Faster R-CNN with EfficientNet-B0 backbone.

    This configuration is used in this study for pneumonia detection
    from chest X-ray images.
    """
    return PneumoniaDetector(
        backbone_name='efficientnet_b0',  # atau 'tf_efficientnet_b0'
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        **kwargs
    )


In [ ]:
# ==============================================================================
# CELL 8: MODEL TESTING
# ==============================================================================

def test_model():
    """Test the model with dummy inputs."""
    print("\n" + "=" * 60)
    print("TESTING MODEL ARCHITECTURE")
    print("=" * 60)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")

    # Create model
    model = create_efficientnet_b0_fasterrcnn(
        num_classes=2,
        pretrained=True
    )
    model = model.to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Test forward pass (training mode)
    print("\n" + "-" * 40)
    print("Testing training forward pass...")
    model.train()

    # Create dummy batch
    batch_size = 2
    images = [torch.randn(3, 1024, 1024).to(device) for _ in range(batch_size)]
    targets = [
        {
            'boxes': torch.tensor([[100, 100, 300, 300], [400, 400, 600, 600]]).float().to(device),
            'labels': torch.tensor([1, 1]).long().to(device),
        }
        for _ in range(batch_size)
    ]

    # Forward pass
    #with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
    # forward pass training atau inference

        losses = model(images, targets)

    print(f"\nTraining losses:")
    for name, loss in losses.items():
        print(f"  {name}: {loss.item():.4f}")

    total_loss = sum(losses.values())
    print(f"  Total: {total_loss.item():.4f}")

    # Test forward pass (inference mode)
    print("\n" + "-" * 40)
    print("Testing inference forward pass...")
    model.eval()

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            detections = model(images)

    print(f"\nInference results (per image):")
    for i, det in enumerate(detections):
        print(f"  Image {i}:")
        print(f"    Boxes: {det['boxes'].shape}")
        print(f"    Labels: {det['labels'].shape}")
        print(f"    Scores: {det['scores'].shape}")
        if len(det['scores']) > 0:
            print(f"    Top score: {det['scores'][0].item():.4f}")

    print("\n" + "=" * 60)
    print("MODEL TEST COMPLETE")
    print("=" * 60)

    return model


def print_model_summary(model: PneumoniaDetector):
    """Print a summary of model components."""
    print("\n" + "=" * 60)
    print("MODEL SUMMARY")
    print("=" * 60)

    print("\n1. BACKBONE (EfficientNet-B0)")
    backbone = model.model.backbone.backbone.body
    for i, (name, module) in enumerate(backbone.named_children()):
        if i < 3:  # Just show first few
            print(f"   {name}: {type(module).__name__}")
    print("   ...")

    print("\n2. FPN (Feature Pyramid Network)")
    fpn = model.model.backbone.fpn
    print(f"   Inner blocks: {len(fpn.inner_blocks)}")
    print(f"   Layer blocks: {len(fpn.layer_blocks)}")

    print("\n3. RPN (Region Proposal Network)")
    rpn = model.model.rpn
    print(f"   Anchor generator sizes: {rpn.anchor_generator.sizes}")

    print("\n4. ROI HEADS")
    roi_heads = model.model.roi_heads
    print(f"   Box predictor: {type(roi_heads.box_predictor).__name__}")

    print("=" * 60)

In [ ]:
# ==============================================================================
# CELL 9: MAIN
# ==============================================================================

if __name__ == "__main__":
    model = test_model()
    print_model_summary(model)

    # Get parameter groups for optimizer
    param_groups = model.get_trainable_parameters(
        lr_backbone=1e-5,
        lr_fpn=5e-5,
        lr_head=1e-4
    )

In [ ]:
#TRAINING ENGINE

In [ ]:
%%writefile data_pipeline_01.py


In [ ]:
%%writefile data_pipeline_01.py
import os
import cv2
import torch
from torch.utils.data import Dataset

class RSNADataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.files = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        # ===== DETEKSI AUGMENTASI =====
        is_hflip = "_hflip" in file
        is_vflip = "_vflip" in file

        pid = file.replace(".jpg","").replace("_hflip","").replace("_vflip","")

        # ===== LOAD IMAGE =====
        img_path = os.path.join(self.img_dir, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Gagal load image: {file}")

        h, w = img.shape
        img = torch.tensor(img).float().unsqueeze(0) / 255.0

        # ===== AMBIL BBOX =====
        rows = self.df[self.df.patientId == pid]

        # ===== HANDLE TANPA BBOX =====
        if len(rows) == 0:
            target = {
                "boxes": torch.zeros((0,4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
            return img, target

        boxes = rows[['x','y','width','height']].values

        # ===== CONVERT KE x1,y1,x2,y2 =====
        boxes[:,2] += boxes[:,0]
        boxes[:,3] += boxes[:,1]

        # ===== FLIP =====
        if is_hflip:
            x_min = boxes[:,0].copy()
            x_max = boxes[:,2].copy()
            boxes[:,0] = w - x_max
            boxes[:,2] = w - x_min

        if is_vflip:
            y_min = boxes[:,1].copy()
            y_max = boxes[:,3].copy()
            boxes[:,1] = h - y_max
            boxes[:,3] = h - y_min

        # ===== CLAMP (ANTI KELUAR GAMBAR) =====
        boxes[:,0] = boxes[:,0].clip(0, w)
        boxes[:,2] = boxes[:,2].clip(0, w)
        boxes[:,1] = boxes[:,1].clip(0, h)
        boxes[:,3] = boxes[:,3].clip(0, h)

        # ===== VALIDASI BBOX =====
        valid_boxes = []
        for box in boxes:
            x1, y1, x2, y2 = box

            if x2 <= x1 or y2 <= y1:
                continue

            valid_boxes.append([x1, y1, x2, y2])

        # ===== JIKA SEMUA INVALID =====
        if len(valid_boxes) == 0:
            print(f"WARNING: bbox invalid di {pid}")

            target = {
                "boxes": torch.zeros((0,4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
            return img, target

        boxes = torch.tensor(valid_boxes).float()

        target = {
            "boxes": boxes,
            "labels": torch.ones(len(boxes), dtype=torch.int64)
        }

        return img, target


def collate_fn(batch):
    images = []
    targets = []

    for img, tgt in batch:
        images.append(img)
        targets.append(tgt)

    return images, targets

In [ ]:
from data_pipeline_01 import RSNADataset, collate_fn
print("data_pipeline OK")


In [ ]:
!sed -n '1,200p' data_pipeline_01.py


In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from data_pipeline_01 import RSNADataset, collate_fn

# Load CSV label
labels_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/stage_2_train_labels.csv"
df = pd.read_csv(labels_path)

# Pakai hanya data pneumonia (punya bounding box)
df_pneumonia = df[df['Target'] == 1]

# Dataset
train_dataset = RSNADataset(
    df=train_df,
    img_dir="/content/train_aug"
)

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# Ambil 1 batch
images, targets = next(iter(train_loader))

print("Jumlah image:", len(images))
print("Keys target:", targets[0].keys())
print("Shape image 0:", images[0].shape)
print("BBox contoh:", targets[0]["boxes"].shape)


In [ ]:
#Buat Data Loader Validasi
from torch.utils.data import DataLoader

val_img_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split/val"

val_dataset = RSNADataset(
    df=val_df,
    img_dir=val_img_dir
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,      # WAJIB untuk evaluasi detection
    shuffle=False,
    collate_fn=collate_fn
)

print("✅ Validation loader ready")


In [ ]:
#mode_arristektur_02.py

In [ ]:
%%writefile model_architecture_02.py
# ==============================================================================
# CELL 1: IMPORTS
# ==============================================================================

from collections import OrderedDict
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# Torchvision detection components
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.ops import FeaturePyramidNetwork
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool

# ==============================================================================
# CELL 2: EFFICIENTNET BACKBONE WRAPPER
# ==============================================================================

class EfficientNetBackbone(nn.Module):
    """
    EfficientNetV2 backbone wrapper for object detection.

    Uses timm's `features_only=True` mode to extract intermediate feature maps
    at multiple scales, which are essential for FPN.

    EfficientNetV2-S Architecture Output Channels:
    - Stage 0: 24 channels  (1/2 resolution)
    - Stage 1: 48 channels  (1/4 resolution)
    - Stage 2: 64 channels  (1/8 resolution)
    - Stage 3: 128 channels (1/16 resolution)
    - Stage 4: 160 channels (1/32 resolution)
    - Stage 5: 256 channels (1/32 resolution)

    We use stages 2-5 for FPN (1/8 to 1/32 resolution) as is standard practice.
    Earlier stages have too high resolution and would be memory-intensive.
    """

    def __init__(
        self,
        model_name: str = 'tf_efficientnet_b0',
        pretrained: bool = True,
        out_indices: Tuple[int, ...] = (2, 3, 4, 5),
        freeze_bn: bool = False
    ):
        """
        Initialize EfficientNet backbone.

        Args:
            model_name: timm model name (tf_efficientnet_b0, tf_efficientnetv2_m, etc.)
            pretrained: Use ImageNet pretrained weights
            out_indices: Which feature stages to output (0-indexed)
            freeze_bn: Freeze BatchNorm layers (useful for small batch sizes)
        """
        super().__init__()

        # Create timm model with feature extraction mode
        self.body = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=out_indices
        )

        # Get output channel dimensions for each stage
        # This is critical for connecting to FPN
        self.out_channels_list = self.body.feature_info.channels()
        self.out_indices = out_indices

        print(f"EfficientNetV2 Backbone initialized:")
        print(f"  Model: {model_name}")
        print(f"  Output stages: {out_indices}")
        print(f"  Output channels: {self.out_channels_list}")

        # Optionally freeze BatchNorm
        # Recommended for batch_size < 4 to prevent BN stats corruption
        if freeze_bn:
            self._freeze_bn()

    def _freeze_bn(self):
        """Freeze all BatchNorm layers."""
        for module in self.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                module.eval()
                for param in module.parameters():
                    param.requires_grad = False

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass returning multi-scale features.

        Args:
            x: Input tensor of shape (B, 3, H, W)

        Returns:
            OrderedDict mapping feature names to tensors
            Keys are '0', '1', '2', '3' corresponding to out_indices
        """
        # Get features from all specified stages
        features = self.body(x)

        # Convert to OrderedDict with string keys (required by FPN)
        out = OrderedDict()
        for idx, feat in enumerate(features):
            out[str(idx)] = feat

        return out

    @property
    def out_channels(self) -> int:
        """Return the number of output channels after FPN (unified)."""
        # FPN unifies all channels to the same dimension
        # We'll set this in the full model
        return 256

# ==============================================================================
# CELL 3: CUSTOM FPN WITH EFFICIENTNET
# ==============================================================================

class EfficientNetWithFPN(nn.Module):
    """
    EfficientNetV2 backbone combined with Feature Pyramid Network.

    WHY FPN?
    ========
    1. Multi-scale detection: Pneumonia opacities vary greatly in size
    2. Combines low-level (edges, textures) and high-level (semantic) features
    3. Standard in all modern object detectors (Faster R-CNN, RetinaNet, YOLO)

    FPN Architecture:
    - Takes multi-scale features from backbone (C2, C3, C4, C5)
    - Creates pyramid features (P2, P3, P4, P5) with same channel dimension
    - Adds top-down pathway with lateral connections
    - Optionally adds P6 via max pooling for very large objects
    """

    def __init__(
        self,
        backbone_name: str = 'tf_efficientnet_b0',
        pretrained: bool = True,
        fpn_out_channels: int = 256,
        freeze_bn: bool = False,
        extra_blocks: bool = False,
    ):
        """
        Initialize backbone + FPN.

        Args:
            backbone_name: timm model name
            pretrained: Use pretrained weights
            fpn_out_channels: Number of channels in FPN output (256 is standard)
            freeze_bn: Freeze BatchNorm layers
            extra_blocks: Add extra max-pool level (P6)
        """
        super().__init__()

        # Create backbone
        self.backbone = EfficientNetBackbone(
            model_name=backbone_name,
            pretrained=pretrained,
            out_indices=(1, 2, 3, 4),  # Use stages with 1/8 to 1/32 resolution
            freeze_bn=freeze_bn
        )

        # Get input channels for FPN from backbone
        in_channels_list = self.backbone.out_channels_list

        # Create FPN
        # extra_blocks adds P6 level for detecting large objects
        extra = LastLevelMaxPool() if extra_blocks else None

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=fpn_out_channels,
            extra_blocks=None
        )

        # Store output channels for detection head
        self.out_channels = fpn_out_channels

        print(f"\nFPN initialized:")
        print(f"  Input channels: {in_channels_list}")
        print(f"  Output channels: {fpn_out_channels}")
        print(f"  Extra P6 level: {extra_blocks}")

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass through backbone and FPN.

        Args:
            x: Input tensor (B, 3, H, W)

        Returns:
            Dictionary of FPN features {'0': P2, '1': P3, '2': P4, '3': P5, 'pool': P6}
        """
        # Extract multi-scale features from backbone
        backbone_features = self.backbone(x)

        # Pass through FPN
        fpn_features = self.fpn(backbone_features)

        return fpn_features


def create_anchor_generator() -> AnchorGenerator:
    """
    Anchor generator adapted from pneumonia X-ray analysis using K-Means++.
    Anchor scales are based on clustered bounding box sizes in the dataset.
    """

    # 3 anchor sizes (derived from dataset statistics)
    # mapped to FPN levels (P3, P4, P5)
    anchor_sizes = (
        (64,),   # medium regions
        (128,),
        (256,),
        (384,),   # large regions
    )

    # Aspect ratios based on article
    aspect_ratios = (
        (0.5, 1.0, 1.25),
    ) * 4

    anchor_generator = AnchorGenerator(
        sizes=anchor_sizes,
        aspect_ratios=aspect_ratios
    )

    return anchor_generator

# ==============================================================================
# CELL 5: ROI POOLER CONFIGURATION
# ==============================================================================

def get_roi_pooler_config() -> Dict:
    """
    Configuration for RoI (Region of Interest) pooling.

    RoI Align is used instead of RoI Pool for better accuracy.
    It uses bilinear interpolation to avoid quantization artifacts.
    """
    return {
        'featmap_names': ['0', '1', '2', '3'],  # FPN levels to use
        'output_size': 7,  # Output spatial size (7x7 is standard)
        'sampling_ratio': 2  # Sampling points for RoI Align
    }


# ==============================================================================
# CELL 6: MAIN PNEUMONIA DETECTOR CLASS
# ==============================================================================

class PneumoniaDetector(nn.Module):
    """
    Complete Pneumonia Detection Model.

    Architecture:
    1. EfficientNetV2-S backbone (timm)
    2. Feature Pyramid Network
    3. Region Proposal Network (RPN)
    4. RoI Align + Detection Head (Faster R-CNN)

    This is a two-stage detector:
    - Stage 1 (RPN): Proposes candidate regions
    - Stage 2 (Detection): Classifies and refines boxes

    Configuration Notes:
    - num_classes=2: Background (0) + Pneumonia (1)
    - Image mean/std: ImageNet normalization (handled in dataset)
    - NMS threshold: 0.5 for RPN, 0.3 for detection (will use WBF post-hoc)
    """

    def __init__(
        self,
        backbone_name: str = 'tf_efficientnet_b0',
        num_classes: int = 2,  # Background + Pneumonia
        pretrained_backbone: bool = True,
        # RPN settings
        rpn_pre_nms_top_n_train: int = 2000,
        rpn_pre_nms_top_n_test: int = 1000,
        rpn_post_nms_top_n_train: int = 1000,
        rpn_post_nms_top_n_test: int = 500,
        rpn_nms_thresh: float = 0.7,
        rpn_fg_iou_thresh: float = 0.7,
        rpn_bg_iou_thresh: float = 0.3,
        # Detection settings
        box_score_thresh: float = 0.05,  # Low threshold - filter with WBF later
        box_nms_thresh: float = 0.5,
        box_detections_per_img: int = 100,
        box_fg_iou_thresh: float = 0.5,
        box_bg_iou_thresh: float = 0.5,
        # Training settings
        freeze_bn: bool = True,  # Freeze BN for small batch sizes
        trainable_backbone_layers: int = 5,  # Fine-tune all layers
    ):
        super().__init__()

        self.num_classes = num_classes
        self.backbone_name = backbone_name

        print("=" * 60)
        print("INITIALIZING PNEUMONIA DETECTOR")
        print("=" * 60)

        # 1. Create backbone with FPN
        backbone_fpn = EfficientNetWithFPN(
            backbone_name=backbone_name,
            pretrained=pretrained_backbone,
            fpn_out_channels=256,
            freeze_bn=freeze_bn,
            extra_blocks=True  # Add P6 level
        )

        # 2. Create anchor generator
        anchor_generator = create_anchor_generator()

        # 3. Create RPN head
        # RPN predicts objectness and box deltas for each anchor
        rpn_head = RPNHead(
            in_channels=backbone_fpn.out_channels,
            num_anchors=anchor_generator.num_anchors_per_location()[0]
        )

        # 4. Create complete Faster R-CNN model
        self.model = FasterRCNN(
            backbone=backbone_fpn,
            num_classes=num_classes,
            # RPN parameters
            rpn_anchor_generator=anchor_generator,
            rpn_head=rpn_head,
            rpn_pre_nms_top_n_train=rpn_pre_nms_top_n_train,
            rpn_pre_nms_top_n_test=rpn_pre_nms_top_n_test,
            rpn_post_nms_top_n_train=rpn_post_nms_top_n_train,
            rpn_post_nms_top_n_test=rpn_post_nms_top_n_test,
            rpn_nms_thresh=rpn_nms_thresh,
            rpn_fg_iou_thresh=rpn_fg_iou_thresh,
            rpn_bg_iou_thresh=rpn_bg_iou_thresh,
            rpn_batch_size_per_image=256,
            rpn_positive_fraction=0.5,
            # Box parameters
            box_score_thresh=box_score_thresh,
            box_nms_thresh=box_nms_thresh,
            box_detections_per_img=box_detections_per_img,
            box_fg_iou_thresh=box_fg_iou_thresh,
            box_bg_iou_thresh=box_bg_iou_thresh,
            box_batch_size_per_image=512,
            box_positive_fraction=0.25,
        )

        print(f"\nFaster R-CNN initialized:")
        print(f"  Num classes: {num_classes}")
        print(f"  RPN NMS thresh: {rpn_nms_thresh}")
        print(f"  Box score thresh: {box_score_thresh}")
        print(f"  Box NMS thresh: {box_nms_thresh}")
        print("=" * 60)

    def forward(
        self,
        images: List[torch.Tensor],
        targets: Optional[List[Dict[str, torch.Tensor]]] = None
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass.

        In training mode:
            Returns loss dictionary {'loss_classifier', 'loss_box_reg',
                                     'loss_objectness', 'loss_rpn_box_reg'}

        In inference mode:
            Returns list of detection dictionaries per image
            [{'boxes': Tensor, 'labels': Tensor, 'scores': Tensor}, ...]

        Args:
            images: List of tensors, each (3, H, W)
            targets: List of target dicts (only needed for training)

        Returns:
            Losses (training) or detections (inference)
        """
        return self.model(images, targets)

    def get_trainable_parameters(
        self,
        lr_backbone: float = 1e-5,
        lr_fpn: float = 1e-4,
        lr_head: float = 1e-3
    ) -> List[Dict]:
        """
        Get parameter groups with different learning rates.

        WHY DIFFERENT LEARNING RATES?
        =============================
        - Backbone: Pretrained, needs small LR to preserve learned features
        - FPN: New layers, can use moderate LR
        - Detection head: New layers, highest LR for fast convergence

        This is called "discriminative learning rates" and is critical
        for fine-tuning pretrained models.
        """
        # Separate parameters by component
        backbone_params = []
        fpn_params = []
        head_params = []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            if 'backbone.backbone' in name:
                backbone_params.append(param)
            elif 'backbone.fpn' in name:
                fpn_params.append(param)
            else:
                head_params.append(param)

        param_groups = [
            {'params': backbone_params, 'lr': lr_backbone, 'name': 'backbone'},
            {'params': fpn_params, 'lr': lr_fpn, 'name': 'fpn'},
            {'params': head_params, 'lr': lr_head, 'name': 'head'},
        ]

        print(f"\nParameter groups:")
        print(f"  Backbone: {len(backbone_params)} params, lr={lr_backbone}")
        print(f"  FPN: {len(fpn_params)} params, lr={lr_fpn}")
        print(f"  Head: {len(head_params)} params, lr={lr_head}")

        return param_groups

    def freeze_backbone(self, freeze: bool = True):
        """Freeze/unfreeze the backbone for transfer learning."""
        for param in self.model.backbone.backbone.parameters():
            param.requires_grad = not freeze
        print(f"Backbone {'frozen' if freeze else 'unfrozen'}")

    def freeze_bn(self):
        """Set all BatchNorm layers to eval mode."""
        for module in self.model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                module.eval()

# ==============================================================================
# CELL 7: MODEL FACTORY (EFFICIENTNET-B0)
# ==============================================================================

def create_efficientnet_b0_fasterrcnn(
    num_classes: int = 2,
    pretrained: bool = True,
    **kwargs
) -> PneumoniaDetector:
    """
    Create Faster R-CNN with EfficientNet-B0 backbone.

    This configuration is used in this study for pneumonia detection
    from chest X-ray images.
    """
    return PneumoniaDetector(
        backbone_name='efficientnet_b0',  # atau 'tf_efficientnet_b0'
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        **kwargs
    )

# ==============================================================================
# CELL 8: MODEL TESTING
# ==============================================================================

def test_model():
    """Test the model with dummy inputs."""
    print("\n" + "=" * 60)
    print("TESTING MODEL ARCHITECTURE")
    print("=" * 60)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")

    # Create model
    model = create_efficientnet_b0_fasterrcnn(
        num_classes=2,
        pretrained=True
    )
    model = model.to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Test forward pass (training mode)
    print("\n" + "-" * 40)
    print("Testing training forward pass...")
    model.train()

    # Create dummy batch
    batch_size = 2
    images = [torch.randn(3, 1024, 1024).to(device) for _ in range(batch_size)]
    targets = [
        {
            'boxes': torch.tensor([[100, 100, 300, 300], [400, 400, 600, 600]]).float().to(device),
            'labels': torch.tensor([1, 1]).long().to(device),
        }
        for _ in range(batch_size)
    ]

    # Forward pass
    #with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
    # forward pass training atau inference

        losses = model(images, targets)

    print(f"\nTraining losses:")
    for name, loss in losses.items():
        print(f"  {name}: {loss.item():.4f}")

    total_loss = sum(losses.values())
    print(f"  Total: {total_loss.item():.4f}")

    # Test forward pass (inference mode)
    print("\n" + "-" * 40)
    print("Testing inference forward pass...")
    model.eval()

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            detections = model(images)

    print(f"\nInference results (per image):")
    for i, det in enumerate(detections):
        print(f"  Image {i}:")
        print(f"    Boxes: {det['boxes'].shape}")
        print(f"    Labels: {det['labels'].shape}")
        print(f"    Scores: {det['scores'].shape}")
        if len(det['scores']) > 0:
            print(f"    Top score: {det['scores'][0].item():.4f}")

    print("\n" + "=" * 60)
    print("MODEL TEST COMPLETE")
    print("=" * 60)

    return model


def print_model_summary(model: PneumoniaDetector):
    """Print a summary of model components."""
    print("\n" + "=" * 60)
    print("MODEL SUMMARY")
    print("=" * 60)

    print("\n1. BACKBONE (EfficientNet-B0)")
    backbone = model.model.backbone.backbone.body
    for i, (name, module) in enumerate(backbone.named_children()):
        if i < 3:  # Just show first few
            print(f"   {name}: {type(module).__name__}")
    print("   ...")

    print("\n2. FPN (Feature Pyramid Network)")
    fpn = model.model.backbone.fpn
    print(f"   Inner blocks: {len(fpn.inner_blocks)}")
    print(f"   Layer blocks: {len(fpn.layer_blocks)}")

    print("\n3. RPN (Region Proposal Network)")
    rpn = model.model.rpn
    print(f"   Anchor generator sizes: {rpn.anchor_generator.sizes}")

    print("\n4. ROI HEADS")
    roi_heads = model.model.roi_heads
    print(f"   Box predictor: {type(roi_heads.box_predictor).__name__}")

    print("=" * 60)

# ==============================================================================
# CELL 9: MAIN
# ==============================================================================

if __name__ == "__main__":
    model = test_model()
    print_model_summary(model)

    # Get parameter groups for optimizer
    param_groups = model.get_trainable_parameters(
        lr_backbone=1e-5,
        lr_fpn=5e-5,
        lr_head=1e-4
    )

In [ ]:
!sed -n '1,200p' model_architecture_02.py

In [ ]:
!ls


In [ ]:
!cp /content/model_architecture_02.py /content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia

In [ ]:
!cat model_architecture_02.py


In [ ]:
!grep -R "create_efficientnet_b0_fasterrcnn" model_architecture_02.py


In [ ]:
from model_architecture_02 import create_efficientnet_b0_fasterrcnn, PneumoniaDetector

model = create_efficientnet_b0_fasterrcnn(num_classes=2)
print(type(model))


In [ ]:
import torch
from data_pipeline_01 import RSNADataset, collate_fn
from torch.utils.data import DataLoader

In [ ]:
# ===== PATH IMAGE =====
train_img_dir = "/content/train_aug"
val_img_dir   = "/content/clahe_split/val"
test_img_dir  = "/content/clahe_split/test"

print(train_img_dir, val_img_dir, test_img_dir)


In [ ]:
train_dataset = RSNADataset(
    df=train_df,
    img_dir=train_img_dir
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,          # kecil dulu (GPU aman)
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

print("Train loader OK")


In [ ]:
import torch
from model_architecture_02 import PneumoniaDetector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)


In [ ]:
images, targets = next(iter(train_loader))

# pindahkan ke device
images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

print("Batch images:", len(images))
print("Image shape :", images[0].shape)
print("Target keys :", targets[0].keys())


In [ ]:
model.train()   # penting: Faster R-CNN loss hanya keluar saat train()

loss_dict = model(images, targets)

print(loss_dict)
print("Total loss:", sum(loss for loss in loss_dict.values()))


In [ ]:
# 1. Ambil grup parameter (Backbone, FPN, Head)
param_groups = model.get_trainable_parameters(
    lr_backbone=1e-5,
    lr_fpn=5e-5,
    lr_head=1e-4
)

# 2. Definisikan RMSprop
# alpha=0.99 adalah nilai standar (smoothing constant)
# momentum=0.9 ditambahkan agar pergerakan gradien lebih stabil
optimizer = torch.optim.RMSprop(
    param_groups,
    alpha=0.99,
    momentum=0.9,
    weight_decay=1e-4
)

print("Optimizer RMSprop OK")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
from tqdm import tqdm
import torch

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        # ===== TANPA AMP =====
        loss_dict = model(images, targets)

        # DEBUG loss per komponen
        for k, v in loss_dict.items():
            if torch.isnan(v):
                print(f"NaN di {k}")

        loss = sum(l for l in loss_dict.values())

        # CEK NaN
        if torch.isnan(loss):
            print("NaN detected, skip batch")
            continue

        loss.backward()

        # 🔥 WAJIB
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)

        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    torch.save(
    {
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict()
    },
    f"/content/drive/MyDrive/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_{epoch+1}.pth"
)

In [ ]:
# Tentukan path file epoch 10 kamu
checkpoint_path = "/content/drive/MyDrive/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_10.pth"

# Load checkpoint
checkpoint = torch.load(checkpoint_path)

# Masukkan bobot ke model
model.load_state_dict(checkpoint['model'])

# Masukkan state optimizer (agar momentum dll tidak reset ke nol)
optimizer.load_state_dict(checkpoint['optimizer'])

# Ambil angka epoch terakhir
start_epoch = checkpoint['epoch']
print(f"Berhasil memuat model. Melanjutkan dari Epoch {start_epoch}")

In [ ]:
from tqdm import tqdm
import torch

# Karena kamu sudah load, start_epoch sekarang adalah 10
# Kita tentukan target akhir 20 epoch
num_epochs = 20

print(f"Memulai pelatihan dari Epoch {start_epoch + 1} hingga {num_epochs}...")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0

    # Deskripsi otomatis menyesuaikan (Misal: Epoch 11/20)
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        # Forward Pass
        loss_dict = model(images, targets)
        loss = sum(l for l in loss_dict.values())

        # Proteksi terhadap NaN
        if torch.isnan(loss):
            print(f"Deteksi NaN pada Epoch {epoch+1}, melewati batch ini.")
            continue

        # Backward Pass
        loss.backward()

        # Gradient Clipping agar RMSProp stabil
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)

        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    # Menghitung rata-rata loss per epoch
    avg_loss = epoch_loss / len(train_loader)
    print(f"Selesai Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

    # SIMPAN CHECKPOINT SETIAP EPOCH
    # Pastikan folder path ini sudah ada di Google Drive kamu
    save_path = f"/content/drive/MyDrive/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_{epoch+1}.pth"

    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict()
    }, save_path)

    print(f"Berhasil menyimpan: {save_path}\n")

In [ ]:
# Tentukan path file epoch 10 kamu
checkpoint_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_17.pth"

# Load checkpoint
checkpoint = torch.load(checkpoint_path)

# Masukkan bobot ke model
model.load_state_dict(checkpoint['model'])

# Masukkan state optimizer (agar momentum dll tidak reset ke nol)
optimizer.load_state_dict(checkpoint['optimizer'])

# Ambil angka epoch terakhir
start_epoch = checkpoint['epoch']
print(f"Berhasil memuat model. Melanjutkan dari Epoch {start_epoch}")

In [ ]:
from tqdm import tqdm
import torch

# Karena kamu sudah load, start_epoch sekarang adalah 10
# Kita tentukan target akhir 20 epoch
num_epochs = 20

print(f"Memulai pelatihan dari Epoch {start_epoch + 1} hingga {num_epochs}...")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0

    # Deskripsi otomatis menyesuaikan (Misal: Epoch 11/20)
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        # Forward Pass
        loss_dict = model(images, targets)
        loss = sum(l for l in loss_dict.values())

        # Proteksi terhadap NaN
        if torch.isnan(loss):
            print(f"Deteksi NaN pada Epoch {epoch+1}, melewati batch ini.")
            continue

        # Backward Pass
        loss.backward()

        # Gradient Clipping agar RMSProp stabil
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)

        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    # Menghitung rata-rata loss per epoch
    avg_loss = epoch_loss / len(train_loader)
    print(f"Selesai Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

    # SIMPAN CHECKPOINT SETIAP EPOCH
    # Pastikan folder path ini sudah ada di Google Drive kamu
    save_path = f"/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_{epoch+1}.pth"

    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict()
    }, save_path)

    print(f"Berhasil menyimpan: {save_path}\n")

In [ ]:
import matplotlib.pyplot as plt

# 1. Data dari hasil training di gambar (Average Loss)
epochs = list(range(1, 21))
average_loss = [
    0.4693, 0.4214, 0.4108, 0.3929, 0.3714,
    0.3394, 0.3258, 0.3068, 0.2937, 0.2830,
    0.2739, 0.2611, 0.2493, 0.2438, 0.2371,
    0.2386, 0.2289, 0.2276, 0.2220, 0.2210
]

# 2. Membuat Grafik
plt.figure(figsize=(10, 6))
plt.plot(epochs, average_loss, marker='o', linestyle='-', color='b', linewidth=2, label='Training Loss')

# 3. Kustomisasi Grafik untuk Laporan Skripsi
plt.title('Grafik Training Loss per Epoch (Optimizer RMSProp)', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.xticks(epochs) # Menampilkan angka 1-10 di sumbu X
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# Menambahkan anotasi nilai pada titik terakhir
plt.annotate(f'{average_loss[-1]}',
             xy=(10, average_loss[-1]),
             xytext=(10.2, average_loss[-1]),
             fontsize=10, color='red')

# 4. Simpan Gambar untuk Bab 4
plt.savefig('grafik_loss_skripsi.png', dpi=300)
plt.show()

In [ ]:
#MENGECEK EPOCH ATAU TRAINING YANG PALING BAGUS

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate_model(model, val_loader, device):
    model.eval()
    all_preds, all_gts = [], []

    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            all_preds.append({
                "boxes": out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu()
            })
            all_gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu()
            })

    return calculate_map(all_preds, all_gts)


In [ ]:
!pip install torchmetrics

In [ ]:
import torch
from tqdm import tqdm
from pprint import pprint
# Perbaikan baris import (harus satu baris atau menggunakan tanda kurung)
from torchmetrics.detection.mean_ap import MeanAveragePrecision

@torch.no_grad()
def evaluate_model(model, val_loader, device):
    model.eval()

    # Inisialisasi metric mAP
    # box_format='xyxy' sesuai output Faster R-CNN
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')

    # Tambahkan tqdm untuk progress bar
    for images, targets in tqdm(val_loader, desc="Evaluating mAP"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        res_preds = []
        res_gts = []

        for out, tgt in zip(outputs, targets):
            # Pastikan data dipindah ke CPU untuk torchmetrics
            res_preds.append({
                "boxes": out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu()
            })
            res_gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu()
            })

        # Update metric per batch
        metric.update(res_preds, res_gts)

    # Hitung hasil akhir
    final_metrics = metric.compute()

    # Sederhanakan output untuk laporan skripsi
    simplified_metrics = {
        "mAP": float(final_metrics["map"]),           # Rata-rata IoU 0.50 s/d 0.95
        "mAP_50": float(final_metrics["map_50"]),     # Standar yang paling sering masuk Bab 4
        "mAP_75": float(final_metrics["map_75"]),     # Indikator lokalisasi yang sangat presisi
        "mAP_small": float(final_metrics["map_small"]) # Akurasi pada lesi pneumonia kecil
    }

    return simplified_metrics

In [ ]:
def calculate_ap(preds, gts, iou_thr):
    TP, FP, FN = 0, 0, 0

    for pred, gt in zip(preds, gts):
        if len(pred["boxes"]) == 0:
            FN += len(gt["boxes"])
            continue

        ious = box_iou(pred["boxes"], gt["boxes"])
        max_iou, _ = ious.max(dim=1)

        TP += (max_iou >= iou_thr).sum().item()
        FP += (max_iou < iou_thr).sum().item()
        FN += max(0, len(gt["boxes"]) - TP)

    precision = TP / (TP + FP + 1e-6)
    recall    = TP / (TP + FN + 1e-6)

    return precision * recall


In [ ]:
from torchvision.ops import box_iou

In [ ]:
def calculate_map(preds, gts, thresholds=[0.4, 0.5, 0.6, 0.7]):
    results = {}

    for thr in thresholds:
        ap = calculate_ap(preds, gts, thr)
        results[f"AP@{thr}"] = ap

    results["mAP"] = sum(results.values()) / len(thresholds)
    return results


In [ ]:
#mAP berdasarkan COCO

In [ ]:
import os
import pandas as pd
from pprint import pprint

EPOCHS_TO_CHECK = range(1, 21)
CHECKPOINT_DIR = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp"

results_list = []

for epoch in EPOCHS_TO_CHECK:
    ckpt_path = f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch}.pth"

    if not os.path.exists(ckpt_path):
        print(f"⚠️ Checkpoint untuk epoch {epoch} tidak ditemukan di: {ckpt_path}")
        continue

    # Load model
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    model.to(device)

    print(f"\n🔍 Menghitung mAP Objektif untuk Epoch {epoch}")

    metrics = evaluate_model(model, val_loader, device)

    # Tambahkan info epoch untuk tabel laporan
    metrics["epoch"] = epoch
    results_list.append(metrics)
    pprint(metrics)

# 3. SIMPAN KE CSV UNTUK EXCEL
if results_list:
    df_results = pd.DataFrame(results_list)
    # Urutkan kolom agar epoch ada di paling depan
    cols = ['epoch'] + [c for c in df_results.columns if c != 'epoch']
    df_results = df_results[cols]

    nama_file = "hasil_evaluasi_mAP_RMSProp.csv"
    df_results.to_csv(nama_file, index=False)
    print(f"\n✅ Berhasil! Semua metrik disimpan dalam file: {nama_file}")
else:
    print("\n❌ Tidak ada data yang berhasil dievaluasi.")

In [ ]:
# Tentukan folder di Drive kamu
NAMA_FILE_LENGKAP = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/hasil_evaluasi_mAP_LR_RMSProp.csv"

# Simpan DataFrame ke Drive
df_results.to_csv(NAMA_FILE_LENGKAP, index=False)
print(f"✅ File berhasil disimpan di Drive: {NAMA_FILE_LENGKAP}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, roc_auc_score, precision_recall_curve,
    auc, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

@torch.no_grad()
def run_final_evaluation(model, data_loader, device, checkpoint_path, threshold=0.5):
    # 1. Load Model dari Checkpoint Terbaik
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    model.eval()

    all_targets = []
    all_scores = []

    # 2. Pengumpulan Data Prediksi
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            # Ground Truth: 1 jika ada pneumonia (box > 0), 0 jika normal
            target_label = 1 if len(targets[i]['boxes']) > 0 else 0
            all_targets.append(target_label)

            # Skor Prediksi: Ambil probabilitas tertinggi dari deteksi
            if len(output['scores']) > 0:
                max_score = torch.max(output['scores']).item()
                all_scores.append(max_score)
            else:
                all_scores.append(0.0)

    y_true = np.array(all_targets)
    y_scores = np.array(all_scores)
    y_pred = (y_scores >= threshold).astype(int)

    # 3. Perhitungan Metrik (Sesuai Standar Paper Saboo et al.)
    metrics = {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Sensitivity)": recall_score(y_true, y_pred, zero_division=0),
    }

    try:
        metrics["AUC-ROC"] = roc_auc_score(y_true, y_scores) [cite: 25]
        precision_pts, recall_pts, _ = precision_recall_curve(y_true, y_scores)
        metrics["AUPRC"] = auc(recall_pts, precision_pts) [cite: 25]
    except:
        metrics["AUC-ROC"] = 0.0
        metrics["AUPRC"] = 0.0

    # 4. Tampilkan Tabel Metrik
    print("\n" + "="*30)
    print("HASIL EVALUASI KLASIFIKASI")
    print("="*30)
    for k, v in metrics.items():
        print(f"{k:20}: {v:.4f}")
    print("="*30)

    # 5. Visualisasi 1: Confusion Matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Pneumonia'])
    disp.plot(cmap=plt.cm.Blues, ax=plt.gca())
    plt.title("Confusion Matrix - Pneumonia Detection")
    plt.show()

    # 6. Visualisasi 2: ROC Curve (Sesuai Figure 3 di Paper)
    plt.figure(figsize=(8, 6))
    RocCurveDisplay.from_predictions(y_true, y_scores, ax=plt.gca())
    plt.plot([0, 1], [0, 1], color='red', linestyle='--') # Garis diagonal bantuan
    plt.title("Receiver Operating Characteristic (ROC) Curve")
    plt.grid(alpha=0.3)
    plt.show()

    return metrics

# --- CARA MENJALANKAN ---
# Ganti path sesuai dengan lokasi file kamu di Google Drive
path_terbaik = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_LR_RMS/checkpoint_epoch_2.pth"
hasil_final = run_final_evaluation(model, val_loader, device, path_terbaik)

In [ ]:
checkpoint_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_2.pth"

In [ ]:
from model_architecture_02 import create_efficientnet_b0_fasterrcnn, PneumoniaDetector

model = create_efficientnet_b0_fasterrcnn(num_classes=2)
print(type(model))

In [ ]:
import torch
from model_architecture_02 import create_efficientnet_b0_fasterrcnn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✅ pakai factory function kamu
model = create_efficientnet_b0_fasterrcnn(num_classes=2)
model.to(device)

# load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# ✅ key sesuai yang kamu save
model.load_state_dict(checkpoint['model'])

# mode inference
model.eval()

In [ ]:
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import os
import numpy as np
from tqdm import tqdm
import cv2
from torch.utils.data import Dataset

In [ ]:
#Penambahan Gambar Normal di Evaluasi untuk mengecek AUC ROC

In [ ]:
class MultiFolderEvalDataset(Dataset):
    def __init__(self, df, dir_pneumonia, dir_normal):
        self.df = df
        self.dir_p = dir_pneumonia
        self.dir_n = dir_normal
        self.pids = df['patientId'].values
        self.targets = df['Target'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        pid = self.pids[idx]
        target_val = self.targets[idx]

        folder = self.dir_p if target_val == 1 else self.dir_n
        img_path = os.path.join(folder, f"{pid}.jpg")

        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((1024, 1024, 3), dtype=np.uint8)
            print(f"⚠️ Warning: {pid} tidak ditemukan di {folder}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = torch.tensor(img).permute(2, 0, 1)

        target = {}
        # Menambahkan real_label agar fungsi evaluasi membacanya dengan mudah
        target['real_label'] = torch.tensor(target_val, dtype=torch.int64)

        if target_val == 1:
            target['boxes'] = torch.tensor([[0, 0, 10, 10]], dtype=torch.float32)
            target['labels'] = torch.tensor([1], dtype=torch.int64)
        else:
            target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
            target['labels'] = torch.tensor([], dtype=torch.int64)

        return img, target

In [ ]:
@torch.no_grad()
def run_final_evaluation_v2(model, data_loader, device, checkpoint_path, threshold=0.5):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    model.eval()

    all_targets = []
    all_scores = []

    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            # Menangani batch_size=1 atau lebih
            target_item = targets[i] if isinstance(targets, list) else targets

            # Ambil label dari dataset
            if 'real_label' in target_item:
                label = target_item['real_label'].item()
            else:
                label = 1 if len(target_item['boxes']) > 0 else 0

            all_targets.append(label)

            if len(output['scores']) > 0:
                all_scores.append(torch.max(output['scores']).item())
            else:
                all_scores.append(0.0)

    y_true = np.array(all_targets)
    y_scores = np.array(all_scores)
    y_pred = (y_scores >= threshold).astype(int)

    print(f"\n📊 Verifikasi Data: Label 0: {np.sum(y_true==0)}, Label 1: {np.sum(y_true==1)}")

    # Hitung metrik standar
    rec = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_scores) if len(np.unique(y_true)) > 1 else 0

    print(f"Recall: {rec:.4f}, Precision: {prec:.4f}, AUC: {auc:.4f}")

    return y_true, y_scores, {"Recall": rec, "Precision": prec, "AUC": auc}

In [ ]:
import os
import pandas as pd

# 1. Tentukan folder
dir_p = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split/val"
dir_n = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/val_normal_clahe"

# 2. Ambil ID yang ada di folder
ids_pneu = [f.replace('.jpg', '') for f in os.listdir(dir_p) if f.endswith('.jpg')]
ids_norm = [f.replace('.jpg', '') for f in os.listdir(dir_n) if f.endswith('.jpg')]

# 3. Buat dataframe Balanced
df_p = pd.DataFrame({'patientId': ids_pneu, 'Target': 1})
df_n = pd.DataFrame({'patientId': ids_norm, 'Target': 0})

# Samakan jumlahnya agar adil (601 vs 601)
min_samples = min(len(df_p), len(df_n))
val_df_sync = pd.concat([df_p.head(min_samples), df_n.head(min_samples)]).reset_index(drop=True)

print(f"✅ Dataframe Sinkron: {len(val_df_sync)} baris")

In [ ]:
from torch.utils.data import DataLoader

# 1. Buat Dataset (Pastikan class MultiFolderEvalDataset sudah di-run di atas)
dataset_sync = MultiFolderEvalDataset(
    val_df_sync,
    dir_pneumonia=dir_p,
    dir_normal=dir_n
)

# 2. Buat Loader (Variabel yang bikin error tadi)
# Gunakan batch_size=1 agar evaluasi biner stabil
val_loader_sync = DataLoader(dataset_sync, batch_size=1, shuffle=False)

# 3. Gunakan Checkpoint Terbaik (Epoch 2)
path_terbaik = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_2.pth"

print("🚀 val_loader_sync SIAP! Sekarang jalankan evaluasi finalnya.")

In [ ]:
# Masukkan ke 3 variabel sekaligus: y_true, y_scores, dan metrics
y_true, y_scores, metrics_final = run_final_evaluation_v2(
    model=model,
    data_loader=val_loader_sync,
    device=device,
    checkpoint_path=path_terbaik
)

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

# --- 1. SETING UKURAN FIGURE (Disesuaikan untuk 1 grafik saja) ---
# Ukuran figsize diubah jadi (8, 6) agar tidak terlalu lebar
fig, ax1 = plt.subplots(1, 1, figsize=(8, 6))

# --- 2. PLOT ROC CURVE (GRAFIK AUC) ---
RocCurveDisplay.from_predictions(
    y_true,
    y_scores,
    name=f"Adam Optimizer (AUC = {metrics_final['AUC']:.2f})",
    ax=ax1,
    color='darkorange',
    linewidth=2
)

# Tambahkan Garis Diagonal (Chance Level)
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance Level (AUC = 0.50)')

# Setting label dan limit
ax1.set_xlim([-0.01, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate (Normal yang salah tebak)')
ax1.set_ylabel('True Positive Rate (Pneumonia yang benar tebak)')
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# --- 1. SETING UKURAN FIGURE (BIAR RAPI DI SKRIPSI) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- 2. PLOT ROC CURVE (GRAFIK AUC) ---
RocCurveDisplay.from_predictions(
    y_true,
    y_scores,
    name=f"RMSProp Optimizer (AUC = {metrics_final['AUC']:.2f})",
    ax=ax1,
    color='darkorange',
    linewidth=2
)
# Tambahkan Garis Diagonal (Chance Level) - INI YANG KAMU CARI
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance Level (AUC = 0.50)')
ax1.set_xlim([-0.01, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# --- 3. PLOT CONFUSION MATRIX ---
# y_pred dibuat dari threshold 0.5
y_pred = (y_scores >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Pneumonia'])
disp.plot(cmap=plt.cm.Blues, ax=ax2, values_format='d')
ax2.set_title('Confusion Matrix - Pneumonia Detection')

plt.tight_layout()
plt.show()

In [ ]:
#Tanpa Mix Data (Hanya Pneumonia)

In [ ]:
#VISUALISASI BACKBONE

In [ ]:
backbone = model.model.backbone.backbone.body
backbone.eval()


In [ ]:
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    features = backbone(img)   # ini list

print(type(features))
print(len(features))


In [ ]:
f2 = features[0][0].cpu()   # stage 2, batch ke-0

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(f2[i], cmap="gray")
    plt.axis("off")

plt.suptitle("EfficientNet Stage 2 (1/8 Resolution)")
plt.show()


In [ ]:
for level, feat in enumerate(features):
    fmap = feat[0].cpu()

    plt.figure(figsize=(12,6))
    for i in range(16):
        plt.subplot(4,4,i+1)
        plt.imshow(fmap[i], cmap="gray")
        plt.axis("off")

    plt.suptitle(f"Feature Map Stage {level+2}")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

img = images[0].unsqueeze(0).to(device)  # 1 gambar
with torch.no_grad():
    stem_out = backbone.conv_stem(img)

feature = stem_out[0].cpu()

plt.figure(figsize=(12,6))
for i in range(16):   # tampilkan 16 channel pertama
    plt.subplot(4,4,i+1)
    plt.imshow(feature[i], cmap="gray")
    plt.axis("off")

plt.suptitle("EfficientNet STEM Features")
plt.show()


In [ ]:
with torch.no_grad():
    features = backbone(img)

# ambil salah satu stage
feat_map = features[2][0].cpu()  # stage tengah

plt.figure(figsize=(12,6))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(feat_map[i], cmap="gray")
    plt.axis("off")

plt.suptitle("Intermediate Convolution Features")
plt.show()


In [ ]:
for i, f in enumerate(features):
    print(f"Stage {i} shape:", f.shape)


In [ ]:
#Visualisasi FPN

In [ ]:
print(model)


In [ ]:
model.eval()
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    backbone_features = model.model.backbone.backbone(img)

for name, feat in backbone_features.items():
    print(f"Level {name}: {feat.shape}")


In [ ]:
with torch.no_grad():
    fpn_features = model.model.backbone(img)

for name, feat in fpn_features.items():
    print(f"P{name}: {feat.shape}")


In [ ]:
import matplotlib.pyplot as plt

model.eval()
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    backbone_features = model.model.backbone.backbone(img)

# Pilih salah satu level, misalnya level terakhir
feature = list(backbone_features.values())[0]  # bisa ubah index 0–3

feature = feature[0].cpu()  # ambil batch pertama

plt.figure(figsize=(10,6))
for i in range(16):  # tampilkan 16 channel pertama
    plt.subplot(4,4,i+1)
    plt.imshow(feature[i], cmap="gray")
    plt.axis("off")

plt.suptitle("Backbone Feature Map")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

model.eval()
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    fpn_features = model.model.backbone(img)

for name, feature in fpn_features.items():
    feature = feature[0].cpu()  # ambil batch pertama

    plt.figure(figsize=(10,6))
    for i in range(16):  # 16 channel pertama
        plt.subplot(4,4,i+1)
        plt.imshow(feature[i], cmap="gray")
        plt.axis("off")

    plt.suptitle(f"FPN Feature Map (P{name})")
    plt.show()


In [ ]:
#SOFT NMS dan EVALUASI

In [ ]:
def soft_nms(
    boxes,
    scores,
    iou_threshold=0.5,
    sigma=0.5,
    score_threshold=0.7
):

    # ✅ HANDLE KASUS TANPA DETEKSI
    if boxes.numel() == 0:
        return boxes, scores

    keep_boxes = []
    keep_scores = []

    boxes = boxes.clone()
    scores = scores.clone()

    while boxes.size(0) > 0:

        max_idx = torch.argmax(scores)
        max_box = boxes[max_idx]
        max_score = scores[max_idx]

        keep_boxes.append(max_box)
        keep_scores.append(max_score)

        boxes = torch.cat([boxes[:max_idx], boxes[max_idx+1:]])
        scores = torch.cat([scores[:max_idx], scores[max_idx+1:]])

        if boxes.size(0) == 0:
            break

        ious = box_iou(max_box.unsqueeze(0), boxes).squeeze(0)
        scores = scores * torch.exp(-(ious ** 2) / sigma)

        keep = scores > score_threshold
        boxes = boxes[keep]
        scores = scores[keep]

    # ✅ HANDLE JIKA SEMUA TERFILTER
    if len(keep_boxes) == 0:
        return torch.empty((0,4)), torch.empty((0,))

    return torch.stack(keep_boxes), torch.stack(keep_scores)


In [ ]:
#Buat Data Loader Validasi
from torch.utils.data import DataLoader

val_img_dir = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/clahe_split/val"

val_dataset = RSNADataset(
    df=val_df,
    img_dir=val_img_dir
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,      # WAJIB untuk evaluasi detection
    shuffle=False,
    collate_fn=collate_fn
)

print("✅ Validation loader ready")


In [ ]:
model.eval()

all_preds = []
all_gts = []

with torch.no_grad():
    for images, targets in tqdm(val_loader):

        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):

            # APPLY SOFT-NMS DI SINI
            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            labels = out["labels"].cpu()[:len(boxes)]

            all_preds.append({
                "boxes": boxes,
                "scores": scores,
                "labels": labels
            })

            all_gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu()
            })


In [ ]:
model.eval()

all_preds = []
all_gts = []

with torch.no_grad():
    for images, targets in tqdm(val_loader):

        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):

            # APPLY SOFT-NMS DI SINI
            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            labels = out["labels"].cpu()[:len(boxes)]

            all_preds.append({
                "boxes": boxes,
                "scores": scores,
                "labels": labels
            })

            all_gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu()
            })


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

def visualize_detection(image, gt_boxes, pred_boxes, scores, score_thr=0.5):

    fig, ax = plt.subplots(1, figsize=(8,8))
    ax.imshow(image, cmap='gray')

    # ===== DRAW GROUND TRUTH (HIJAU) =====
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor='green',
            facecolor='none'
        )
        ax.add_patch(rect)

    # ===== DRAW PREDICTION (MERAH) =====
    for box, score in zip(pred_boxes, scores):
        if score >= score_thr:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor='red',
                facecolor='none'
            )
            ax.add_patch(rect)

            ax.text(
                x1, y1,
                f"{score:.2f}",
                color='red',
                fontsize=10,
                backgroundcolor='white'
            )

    plt.axis('off')
    plt.show()

In [ ]:
#Tambahan Titik Kordinat
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_detection(image, gt_boxes=None, pred_boxes=None, scores=None, score_thr=0.5):
    """
    Fungsi visualisasi fleksibel untuk data RSNA (dengan GT)
    maupun data eksternal (tanpa GT).
    """
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image, cmap='gray')

    # ===== DRAW GROUND TRUTH (HIJAU) =====
    # Cek apakah gt_boxes ada (tidak None) dan tidak kosong
    if gt_boxes is not None and len(gt_boxes) > 0:
        for box in gt_boxes:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor='green',
                facecolor='none',
                label='Ground Truth' # Tambahkan label untuk legenda jika perlu
            )
            ax.add_patch(rect)

    # ===== DRAW PREDICTION (MERAH) =====
    # Cek apakah pred_boxes ada
    if pred_boxes is not None and scores is not None:
        for box, score in zip(pred_boxes, scores):
            if score >= score_thr:
                x1, y1, x2, y2 = box
                rect = patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none'
                )
                ax.add_patch(rect)

                ax.text(
                    x1, y1 - 5, # Digeser sedikit ke atas kotak agar rapi
                    f"{score:.2f}",
                    color='red',
                    fontsize=10,
                    weight='bold',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
                )

    plt.axis('off')
    plt.tight_layout()
    return fig # Mengembalikan figure agar bisa ditampilkan di Streamlit via st.pyplot(fig)

In [ ]:
model.eval()

with torch.no_grad():
    for images, targets in val_loader:

        images = [img.to(device) for img in images]
        outputs = model(images)

        for img, out, tgt in zip(images, outputs, targets):

            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            visualize_detection(
                image=img.permute(1,2,0).cpu(),  # ⭐ FIX DI SINI
                gt_boxes=tgt["boxes"],
                pred_boxes=boxes,
                scores=scores,
                score_thr=0.5
            )


In [ ]:
model.eval()

with torch.no_grad():
    for images, targets in val_loader:

        images = [img.to(device) for img in images]
        outputs = model(images)

        for img, out, tgt in zip(images, outputs, targets):

            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            # 🔥 FILTER BERDASARKAN SCORE THRESHOLD
            keep = scores >= 0.5
            boxes = boxes[keep]
            scores = scores[keep]

            # 🔥 CEK JUMLAH BOUNDING BOX
            if len(boxes) == 4:

                print(f"Ditemukan gambar dengan 4 bounding box")

                visualize_detection(
                    image=img.permute(1,2,0).cpu(),
                    gt_boxes=tgt["boxes"],
                    pred_boxes=boxes,
                    scores=scores,
                    score_thr=0.5
                )

                break  # hentikan setelah ketemu satu (opsional)

In [ ]:
def compute_ap(preds, gts, iou_thr):
    tp, fp, scores = [], [], []

    for pred, gt in zip(preds, gts):
        if len(pred["boxes"]) == 0:
            continue

        ious = box_iou(pred["boxes"], gt["boxes"])
        max_iou, _ = ious.max(dim=1)

        for iou, score in zip(max_iou, pred["scores"]):
            scores.append(score.item())
            if iou >= iou_thr:
                tp.append(1)
                fp.append(0)
            else:
                tp.append(0)
                fp.append(1)

    if len(tp) == 0:
        return 0.0

    tp = torch.tensor(tp)
    fp = torch.tensor(fp)
    scores = torch.tensor(scores)

    idx = torch.argsort(scores, descending=True)
    tp, fp = tp[idx], fp[idx]

    tp = torch.cumsum(tp, 0)
    fp = torch.cumsum(fp, 0)

    recall = tp / (tp[-1] + 1e-6)
    precision = tp / (tp + fp + 1e-6)

    ap = torch.trapz(precision, recall).item()
    return ap


In [ ]:
#score threshold 0.3
iou_thresholds = [0.4, 0.5, 0.6, 0.7]
ap_results = {}

for thr in iou_thresholds:
    ap = compute_ap(all_preds, all_gts, thr)
    ap_results[f"AP@{thr}"] = ap
    print(f"AP@{thr}: {ap:.4f}")


In [ ]:
map_value = sum(ap_results.values()) / len(ap_results)
print(f"mAP: {map_value:.4f}")


In [ ]:
#score threshold 0.4
iou_thresholds = [0.4, 0.5, 0.6, 0.7]
ap_results = {}

for thr in iou_thresholds:
    ap = compute_ap(all_preds, all_gts, thr)
    ap_results[f"AP@{thr}"] = ap
    print(f"AP@{thr}: {ap:.4f}")


In [ ]:
map_value = sum(ap_results.values()) / len(ap_results)
print(f"mAP: {map_value:.4f}")


In [ ]:
#score threshold 0.5
iou_thresholds = [0.4, 0.5, 0.6, 0.7]
ap_results = {}

for thr in iou_thresholds:
    ap = compute_ap(all_preds, all_gts, thr)
    ap_results[f"AP@{thr}"] = ap
    print(f"AP@{thr}: {ap:.4f}")


In [ ]:
map_value = sum(ap_results.values()) / len(ap_results)
print(f"mAP: {map_value:.4f}")


In [ ]:
#score threshold 0.6
iou_thresholds = [0.4, 0.5, 0.6, 0.7]
ap_results = {}

for thr in iou_thresholds:
    ap = compute_ap(all_preds, all_gts, thr)
    ap_results[f"AP@{thr}"] = ap
    print(f"AP@{thr}: {ap:.4f}")


In [ ]:
map_value = sum(ap_results.values()) / len(ap_results)
print(f"mAP: {map_value:.4f}")


In [ ]:
#score threshold 0.7
iou_thresholds = [0.4, 0.5, 0.6, 0.7]
ap_results = {}

for thr in iou_thresholds:
    ap = compute_ap(all_preds, all_gts, thr)
    ap_results[f"AP@{thr}"] = ap
    print(f"AP@{thr}: {ap:.4f}")


In [ ]:
map_value = sum(ap_results.values()) / len(ap_results)
print(f"mAP: {map_value:.4f}")


In [ ]:
import pandas as pd

df_eval = pd.DataFrame([ap_results])
df_eval["mAP"] = map_value
df_eval


In [ ]:
#MENGHITUNG RECALL, PRECISION, ROC CURVE, CONFUSION MATRIX

In [ ]:
def detection_stats(preds, gts, iou_thr=0.5, score_thr=0.7):

    TP, FP, FN = 0, 0, 0
    y_true = []
    y_scores = []

    for pred, gt in zip(preds, gts):

        # filter score
        keep = pred["scores"] >= score_thr
        pred_boxes = pred["boxes"][keep]
        pred_scores = pred["scores"][keep]

        gt_boxes = gt["boxes"]

        if len(pred_boxes) == 0:
            FN += len(gt_boxes)
            continue

        if len(gt_boxes) == 0:
            FP += len(pred_boxes)
            continue

        ious = box_iou(pred_boxes, gt_boxes)
        max_iou, _ = ious.max(dim=1)

        for iou, score in zip(max_iou, pred_scores):
            y_scores.append(score.item())

            if iou >= iou_thr:
                TP += 1
                y_true.append(1)
            else:
                FP += 1
                y_true.append(0)

        FN += max(0, len(gt_boxes) - (max_iou >= iou_thr).sum().item())

    return TP, FP, FN, y_true, y_scores


In [ ]:
#PRECISION & RECALL (ST=0.3)
TP, FP, FN, y_true, y_scores = detection_stats(all_preds, all_gts)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


In [ ]:
#PRECISION & RECALL (ST=0.4)
TP, FP, FN, y_true, y_scores = detection_stats(all_preds, all_gts)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


In [ ]:
#PRECISION & RECALL (ST=0.5)
TP, FP, FN, y_true, y_scores = detection_stats(all_preds, all_gts)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


In [ ]:
#PRECISION & RECALL (ST=0.6)
TP, FP, FN, y_true, y_scores = detection_stats(all_preds, all_gts)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


In [ ]:
#PRECISION & RECALL (ST=0.7)
TP, FP, FN, y_true, y_scores = detection_stats(all_preds, all_gts)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


In [ ]:
#CONFUSION MATRIX
import numpy as np
import matplotlib.pyplot as plt

cm = np.array([
    [TP, FN],
    [FP, 0]
])

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Detection Confusion Matrix")
plt.colorbar()

plt.xticks([0,1], ["Detected","Missed"])
plt.yticks([0,1], ["Ground Truth","False Detection"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.show()



In [ ]:
#ROC CURVE
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
#Pengujian Data Tes set

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T


In [ ]:
test_img_dir = "/content/clahe_split/test"


In [ ]:
print([name for name in dir() if "Dataset" in name])

In [ ]:
test_img_dir = "/content/clahe_split/test"

test_dataset = RSNADataset(test_df, test_img_dir)

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
img, target = test_dataset[0]

print(img.shape)
print(target)

In [ ]:
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):

            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            # filter pakai threshold terbaik
            keep = scores >= 0.5
            boxes = boxes[keep]
            scores = scores[keep]

            all_predictions.append({
                "boxes": boxes,
                "scores": scores
            })

            all_targets.append(tgt)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision.ops as ops

def visualize_all_test(model, loader, device, score_thr=0.5):
    model.eval()

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for img, out, tgt in zip(images, outputs, targets):

                boxes, scores = soft_nms(
                    out["boxes"].cpu(),
                    out["scores"].cpu()
                )

                keep = scores >= score_thr
                boxes = boxes[keep]

                fig, ax = plt.subplots(1, figsize=(6,6)) #figsizenya ubah ke 8x8 biar lebih kelihatan
                #ax.axis('off') -> pake kode ini untuk menghilangkan matriksnya
                ax.imshow(img.permute(1,2,0).cpu(), cmap="gray")

                # Ground Truth (Hijau)
                for box in tgt["boxes"]:
                    x1, y1, x2, y2 = box
                    rect = patches.Rectangle(
                        (x1, y1),
                        x2-x1,
                        y2-y1,
                        linewidth=2,
                        edgecolor='green',
                        facecolor='none'
                    )
                    ax.add_patch(rect)

                # Prediksi (Merah)
                for box in boxes:
                    x1, y1, x2, y2 = box
                    rect = patches.Rectangle(
                        (x1, y1),
                        x2-x1,
                        y2-y1,
                        linewidth=2,
                        edgecolor='red',
                        facecolor='none'
                    )
                    ax.add_patch(rect)

                plt.show()

In [ ]:
visualize_all_test(model, test_loader, device)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_4_predictions(model, loader, device, score_thr=0.5):
    model.eval()

    with torch.no_grad():
        for images, targets in loader:

            images = [img.to(device) for img in images]
            outputs = model(images)

            for img, out, tgt in zip(images, outputs, targets):

                boxes, scores = soft_nms(
                    out["boxes"].cpu(),
                    out["scores"].cpu()
                )

                keep = scores >= score_thr
                boxes = boxes[keep]
                scores = scores[keep]

                # 🔥 Filter: hanya tampilkan yang punya 4 bbox
                if len(boxes) == 4:

                    fig, ax = plt.subplots(1, figsize=(6,6))
                    ax.imshow(img.permute(1,2,0).cpu(), cmap="gray")

                    # Prediksi → merah
                    for box, score in zip(boxes, scores):
                        x1, y1, x2, y2 = box
                        rect = patches.Rectangle(
                            (x1, y1),
                            x2-x1,
                            y2-y1,
                            linewidth=2,
                            edgecolor='red',
                            facecolor='none'
                        )
                        ax.add_patch(rect)
                        ax.text(x1, y1-5,
                                f"{score:.2f}",
                                color='red',
                                fontsize=8)

                    # Ground Truth → hijau
                    for gt in tgt["boxes"]:
                        x1, y1, x2, y2 = gt
                        rect = patches.Rectangle(
                            (x1, y1),
                            x2-x1,
                            y2-y1,
                            linewidth=2,
                            edgecolor='green',
                            facecolor='none'
                        )
                        ax.add_patch(rect)

                    plt.title("Pred (Red) | GT (Green)")
                    plt.show()

In [ ]:
visualize_4_predictions(model, test_loader, device)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_4_predictions(model, loader, device, score_thr=0.5):
    model.eval()

    with torch.no_grad():
        for images, targets in loader:

            images = [img.to(device) for img in images]
            outputs = model(images)

            for img, out, tgt in zip(images, outputs, targets):

                boxes, scores = soft_nms(
                    out["boxes"].cpu(),
                    out["scores"].cpu()
                )

                keep = scores >= score_thr
                boxes = boxes[keep]
                scores = scores[keep]

                # 🔥 Filter: hanya tampilkan yang punya 4 bbox
                if len(boxes) == 3:

                    fig, ax = plt.subplots(1, figsize=(6,6))
                    ax.imshow(img.permute(1,2,0).cpu(), cmap="gray")

                    # Prediksi → merah
                    for box, score in zip(boxes, scores):
                        x1, y1, x2, y2 = box
                        rect = patches.Rectangle(
                            (x1, y1),
                            x2-x1,
                            y2-y1,
                            linewidth=2,
                            edgecolor='red',
                            facecolor='none'
                        )
                        ax.add_patch(rect)
                        ax.text(x1, y1-5,
                                f"{score:.2f}",
                                color='red',
                                fontsize=8)

                    # Ground Truth → hijau
                    for gt in tgt["boxes"]:
                        x1, y1, x2, y2 = gt
                        rect = patches.Rectangle(
                            (x1, y1),
                            x2-x1,
                            y2-y1,
                            linewidth=2,
                            edgecolor='green',
                            facecolor='none'
                        )
                        ax.add_patch(rect)

                    plt.title("Pred (Red) | GT (Green)")
                    plt.show()

In [ ]:
visualize_4_predictions(model, test_loader, device)

In [ ]:
from torchvision.ops import box_iou

def evaluate_detection(model, loader, device, score_thr=0.5, iou_thr=0.5):
    model.eval()

    TP = 0
    FP = 0
    FN = 0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for out, tgt in zip(outputs, targets):

                boxes, scores = soft_nms(
                    out["boxes"].cpu(),
                    out["scores"].cpu()
                )

                keep = scores >= score_thr
                pred_boxes = boxes[keep]

                gt_boxes = tgt["boxes"]

                if len(pred_boxes) == 0:
                    FN += len(gt_boxes)
                    continue

                ious = box_iou(pred_boxes, gt_boxes)

                matched_gt = set()

                for i in range(len(pred_boxes)):
                    max_iou, idx = ious[i].max(0)
                    if max_iou >= iou_thr and idx.item() not in matched_gt:
                        TP += 1
                        matched_gt.add(idx.item())
                    else:
                        FP += 1

                FN += len(gt_boxes) - len(matched_gt)

    return TP, FP, FN

In [ ]:
TP, FP, FN = evaluate_detection(model, test_loader, device)
print("TP:", TP)
print("FP:", FP)
print("FN:", FN)

In [ ]:
#CONFUSION MATRIX
import numpy as np
import matplotlib.pyplot as plt

cm = np.array([
    [TP, FN],
    [FP, 0]
])

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Detection Confusion Matrix")
plt.colorbar()

plt.xticks([0,1], ["Detected","Missed"])
plt.yticks([0,1], ["Ground Truth","False Detection"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.show()



In [ ]:
precision = TP / (TP + FP + 1e-6)
recall    = TP / (TP + FN + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

In [ ]:
#roc curve auc
from torchvision.ops import box_iou

def collect_pr_data(model, loader, device, iou_thr=0.5):
    model.eval()

    all_scores = []
    all_labels = []

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for out, tgt in zip(outputs, targets):

                boxes = out["boxes"].cpu()
                scores = out["scores"].cpu()
                gt_boxes = tgt["boxes"]

                if len(boxes) == 0:
                    continue

                if len(gt_boxes) == 0:
                    # Semua prediksi jadi FP
                    all_scores.extend(scores.tolist())
                    all_labels.extend([0]*len(scores))
                    continue

                ious = box_iou(boxes, gt_boxes)
                matched_gt = set()

                for i in range(len(boxes)):
                    max_iou, idx = ious[i].max(0)

                    if max_iou >= iou_thr and idx.item() not in matched_gt:
                        all_labels.append(1)  # TP
                        matched_gt.add(idx.item())
                    else:
                        all_labels.append(0)  # FP

                    all_scores.append(scores[i].item())

    return all_labels, all_scores

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

y_true, y_scores = collect_pr_data(model, test_loader, device)

precision, recall, thresholds = precision_recall_curve(y_true, y_scores)

plt.figure()
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_true, y_scores)
print("Average Precision (AP):", ap)

In [ ]:
#PERCOBAAN DENGAN DATA UJI BARU (DATA EKSTERNAL) //PERBAIKI LAGI MAISH ERROR

In [ ]:
checkpoint_path = '/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/pneumonia_RMSProp/checkpoint_epoch_2.pth'

In [ ]:
from model_architecture_02 import create_efficientnet_b0_fasterrcnn, PneumoniaDetector

model = create_efficientnet_b0_fasterrcnn(num_classes=2)
print(type(model))

In [ ]:
import torch
from model_architecture_02 import create_efficientnet_b0_fasterrcnn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✅ pakai factory function kamu
model = create_efficientnet_b0_fasterrcnn(num_classes=2)
model.to(device)

# load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# ✅ key sesuai yang kamu save
model.load_state_dict(checkpoint['model'])

# mode inference
model.eval()

In [ ]:
class ExternalDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.files = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, file), 0)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        img = torch.tensor(img).permute(2,0,1).float() / 255.0

        return img, file


In [ ]:
import os
import cv2
import torch

class ExternalDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, img_size=1024):
        self.img_dir = img_dir
        self.img_size = img_size
        self.files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".png", ".bmp"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        # baca grayscale
        img = cv2.imread(os.path.join(self.img_dir, file), 0)

        # resize ke 1024
        img = cv2.resize(img, (self.img_size, self.img_size))

        # grayscale → RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # ke tensor
        img = torch.tensor(img).permute(2,0,1).float() / 255.0

        return img, file

In [ ]:
import os
import cv2
import torch

class ExternalDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, img_size=1024):
        self.img_dir = img_dir
        self.img_size = img_size
        self.files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".png", ".bmp"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        # baca grayscale
        img = cv2.imread(os.path.join(self.img_dir, file), 0)

        # resize ke 1024
        img = cv2.resize(img, (self.img_size, self.img_size))

        # grayscale → RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # ke tensor
        img = torch.tensor(img).permute(2,0,1).float() / 255.0

        return img, file

In [ ]:
external_loader = DataLoader(
    ExternalDataset("/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/Pneumonia_data_eksternal/images"),
    batch_size=1,
    shuffle=False
)

results = []

with torch.no_grad():
    for images, names in tqdm(external_loader):

        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, name in zip(outputs, names):

            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            results.append({
                "image": name,
                "boxes": boxes,
                "scores": scores
            })


In [ ]:
from tqdm import tqdm
external_loader = DataLoader(
    ExternalDataset("/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/Pneumonia_data_eksternal/images"),
    batch_size=1,
    shuffle=False
)

results = []

with torch.no_grad():
    for images, names in tqdm(external_loader):

        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, name in zip(outputs, names):

            boxes, scores = soft_nms(
                out["boxes"].cpu(),
                out["scores"].cpu()
            )

            results.append({
                "image": name,
                "boxes": boxes,
                "scores": scores
            })


In [ ]:
for images, filenames in external_loader:
    for img in images:
        print(img.shape)
    break

In [ ]:
import os
import cv2
import torch

class ExternalDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, img_size=1024):
        self.img_dir = img_dir
        self.img_size = img_size
        self.files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".png", ".bmp"))
        ])

        # buat CLAHE object
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        # baca grayscale
        img = cv2.imread(os.path.join(self.img_dir, file), cv2.IMREAD_GRAYSCALE)

        # ===== CLAHE =====
        img = self.clahe.apply(img)

        # ===== RESIZE =====
        img = cv2.resize(img, (self.img_size, self.img_size))

        # grayscale → RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # ke tensor
        img = torch.tensor(img).permute(2,0,1).float() / 255.0

        return img, file

In [ ]:
model.eval()

In [ ]:
score_thr = 0.3

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        pred_label = "PNEUMONIA"
    else:
        pred_label = "Tidak ditemukan pneumonia"

    print(r["image"], pred_label)


In [ ]:
score_thr = 0.5

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        pred_label = "PNEUMONIA"
    else:
        pred_label = "Tidak ditemukan pneumonia"

    print(r["image"], pred_label)


In [ ]:
for r in results:
    if len(r["scores"]) > 0:
        print(r["scores"].max().item())

In [ ]:
score_thr = 0.3
jumlah_pneumonia = 0
jumlah_normal = 0

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        jumlah_pneumonia += 1
    else:
        jumlah_normal += 1

print("Total PNEUMONIA            :", jumlah_pneumonia)
print("Tidak ditemukan pneumonia  :", jumlah_normal)
print("Total gambar               :", jumlah_pneumonia + jumlah_normal)


In [ ]:
score_thr = 0.5
jumlah_pneumonia = 0
jumlah_normal = 0

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        jumlah_pneumonia += 1
    else:
        jumlah_normal += 1

print("Total PNEUMONIA            :", jumlah_pneumonia)
print("Tidak ditemukan pneumonia  :", jumlah_normal)
print("Total gambar               :", jumlah_pneumonia + jumlah_normal)


In [ ]:
total = jumlah_pneumonia + jumlah_normal

print(f"PNEUMONIA : {jumlah_pneumonia} ({jumlah_pneumonia/total*100:.2f}%)")
print(f"NORMAL    : {jumlah_normal} ({jumlah_normal/total*100:.2f}%)")


In [ ]:
score_thr = 0.3

TP = 0   # pneumonia berhasil terdeteksi
FN = 0   # pneumonia tidak terdeteksi

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        TP += 1
    else:
        FN += 1

total = TP + FN
sensitivity = TP / (total + 1e-6)

print(f"Total data eksternal : {total}")
print(f"Terdeteksi pneumonia (TP) : {TP}")
print(f"Tidak terdeteksi (FN) : {FN}")
print(f"Sensitivity : {sensitivity:.4f}")


In [ ]:
score_thr = 0.5

TP = 0   # pneumonia berhasil terdeteksi
FN = 0   # pneumonia tidak terdeteksi

for r in results:

    if len(r["scores"]) > 0 and r["scores"].max() >= score_thr:
        TP += 1
    else:
        FN += 1

total = TP + FN
sensitivity = TP / (total + 1e-6)

print(f"Total data eksternal : {total}")
print(f"Terdeteksi pneumonia (TP) : {TP}")
print(f"Tidak terdeteksi (FN) : {FN}")
print(f"Sensitivity : {sensitivity:.4f}")


In [ ]:
all_scores = []

for r in results:
    if len(r["scores"]) > 0:
        all_scores.append(r["scores"].max().item())

plt.hist(all_scores, bins=20)
plt.title("Confidence Distribution - External Data")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import os

def visualize_external(image_path, boxes, scores, score_thr=0.3):

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 🔴 TAMBAHKAN INI
    img = cv2.resize(img, (1024, 1024))

    fig, ax = plt.subplots(1, figsize=(6,6))
    ax.imshow(img)

    for box, score in zip(boxes, scores):

        if score < score_thr:
            continue

        x1, y1, x2, y2 = box

        rect = patches.Rectangle(
            (x1, y1),
            x2-x1,
            y2-y1,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )

        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{score:.2f}", color="yellow")

    plt.axis("off")
    plt.show()

In [ ]:
base_path = "/content/drive/Shareddrives/tatatrinovita/RSNA_Pneumonia/Pneumonia_data_eksternal/images"

for r in results:
    img_path = os.path.join(base_path, r["image"])

    visualize_external(
        img_path,
        r["boxes"],
        r["scores"],
        score_thr=0.5
    )


In [ ]:
#VISUALISASI UNTUK KEPERLUAN RPN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

folder_path = "/content/drive/MyDrive/RSNA_Pneumonia/train_jpg"
print(os.listdir(folder_path)[:10])  # tampilkan 10 file pertama


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === Path gambar ===
img_path = "/content/drive/MyDrive/RSNA_Pneumonia/train_jpg/ec910555-f698-4536-aff7-7fa9d3405d60.jpg"

# === Load gambar ===
img = plt.imread(img_path)

# Cek ukuran gambar
print("Shape gambar:", img.shape)

# Jika grayscale
if len(img.shape) == 2:
    h, w = img.shape
else:
    h, w, _ = img.shape

# === Tentukan satu titik (tengah gambar) ===
center_x = w // 2
center_y = h // 2

# === Anchor sesuai penelitian kamu ===
sizes = [64, 128, 256, 384]   # 4 level FPN
ratios = [0.5, 1.0, 1.25]     # 3 rasio aspek

# === Plot ===
fig, ax = plt.subplots(1, figsize=(6,6))
ax.imshow(img, cmap='gray')

for size in sizes:
    for ratio in ratios:
        anchor_w = size * ratio
        anchor_h = size / ratio

        rect = patches.Rectangle(
            (center_x - anchor_w/2, center_y - anchor_h/2),
            anchor_w,
            anchor_h,
            linewidth=1.5,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)

plt.title("Visualisasi Anchor pada Satu Titik Grid")
plt.axis("off")
plt.show()


In [ ]:
sizes = [128]   # satu level saja
ratios = [0.5, 1.0, 1.25]


In [ ]:
colors = ['red', 'green', 'blue', 'yellow']

for i, size in enumerate(sizes):
    for ratio in ratios:
        anchor_w = size * ratio
        anchor_h = size / ratio

        rect = patches.Rectangle(
            (center_x - anchor_w/2, center_y - anchor_h/2),
            anchor_w,
            anchor_h,
            linewidth=2,
            edgecolor=colors[i],
            facecolor='none'
        )
        ax.add_patch(rect)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === Load gambar ===
img_path = "/content/drive/MyDrive/RSNA_Pneumonia/train_jpg/ec910555-f698-4536-aff7-7fa9d3405d60.jpg"
img = plt.imread(img_path)

# Ambil ukuran gambar
if len(img.shape) == 2:
    h, w = img.shape
else:
    h, w, _ = img.shape

center_x = w // 2
center_y = h // 2

# === Anchor parameter penelitian ===
sizes = [64, 128, 256, 384]
ratios = [0.5, 1.0, 1.25]
colors = ['red', 'green', 'blue', 'yellow']

# === Plot ===
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(img, cmap='gray')

for i, size in enumerate(sizes):
    for ratio in ratios:
        anchor_w = size * ratio
        anchor_h = size / ratio

        rect = patches.Rectangle(
            (center_x - anchor_w/2, center_y - anchor_h/2),
            anchor_w,
            anchor_h,
            linewidth=2,
            edgecolor=colors[i],
            facecolor='none'
        )
        ax.add_patch(rect)

plt.title("Visualisasi Anchor Multi-Scale pada Satu Titik")
plt.axis("off")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === Load gambar ===
img_path = "/content/drive/MyDrive/RSNA_Pneumonia/train_jpg/ec910555-f698-4536-aff7-7fa9d3405d60.jpg"
img = plt.imread(img_path)

# Ambil ukuran gambar
if len(img.shape) == 2:
    h, w = img.shape
else:
    h, w, _ = img.shape

center_x = w // 2
center_y = h // 2

# === Anchor parameter penelitian ===
sizes = [64, 128, 256, 512]
ratios = [0.5, 1.0, 2.0]
colors = ['red', 'green', 'blue', 'yellow']

# === Plot ===
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(img, cmap='gray')

for i, size in enumerate(sizes):
    for ratio in ratios:
        anchor_w = size * ratio
        anchor_h = size / ratio

        rect = patches.Rectangle(
            (center_x - anchor_w/2, center_y - anchor_h/2),
            anchor_w,
            anchor_h,
            linewidth=2,
            edgecolor=colors[i],
            facecolor='none'
        )
        ax.add_patch(rect)

plt.title("Visualisasi Anchor Multi-Scale pada Satu Titik")
plt.axis("off")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

img_path = "/content/drive/MyDrive/RSNA_Pneumonia/train_jpg/ec910555-f698-4536-aff7-7fa9d3405d60.jpg"
img = plt.imread(img_path)

if len(img.shape) == 2:
    h, w = img.shape
else:
    h, w, _ = img.shape

center_x = w // 2
center_y = h // 2

# Misalnya level P3 → size 128
size = 128
ratios = [0.5, 1.0, 1.25]

fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(img, cmap='gray')

for ratio in ratios:
    anchor_w = size * ratio
    anchor_h = size / ratio

    rect = patches.Rectangle(
        (center_x - anchor_w/2, center_y - anchor_h/2),
        anchor_w,
        anchor_h,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

plt.title("Visualisasi Anchor pada Satu Grid (Level P3)")
plt.axis("off")
plt.show()


In [ ]:
print(type(img))


In [ ]:
print(img.shape)


In [ ]:
import torch
import numpy as np

if len(img.shape) == 2:  # grayscale
    img = np.stack([img, img, img], axis=-1)

img_tensor = torch.from_numpy(img).float().permute(2,0,1) / 255.0


In [ ]:
#MENAMPILKAN PROPOSAL RPN SAJA (SEBELUM ROI HEAD)
model.model.eval()


In [ ]:
next(model.model.parameters()).device


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.model.to(device)
img_tensor = img_tensor.to(device)


In [ ]:
model.model.eval()

with torch.no_grad():
    images_list, _ = model.model.transform([img_tensor])
    features = model.model.backbone(images_list.tensors)
    proposals, _ = model.model.rpn(images_list, features)

print("Jumlah proposal sebelum ROI:", proposals[0].shape)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

img_np = img_tensor.permute(1,2,0).cpu().numpy()

fig, ax = plt.subplots(1, figsize=(8,8))
ax.imshow(img_np)

# tampilkan 200 box saja supaya tidak terlalu penuh
for box in proposals[0][:200]:
    x1, y1, x2, y2 = box.cpu().numpy()
    rect = patches.Rectangle((x1,y1),
                             x2-x1,
                             y2-y1,
                             linewidth=1,
                             edgecolor='red',
                             facecolor='none')
    ax.add_patch(rect)

plt.title("Proposal RPN (Sebelum ROI)")
plt.show()


In [ ]:
feature_maps = list(features.values())


In [ ]:
anchors = rpn.anchor_generator(images_list, feature_maps)


In [ ]:
rpn = model.model.rpn

# 1️⃣ Ambil feature map dari backbone
features = model.model.backbone(images_list.tensors)

# 2️⃣ Ubah ke list tensor (P2–P5)
feature_maps = list(features.values())

# 3️⃣ Generate anchor
anchors = rpn.anchor_generator(images_list, feature_maps)

print("Jumlah anchor total:", sum(a.shape[0] for a in anchors))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

img_np = img_tensor.permute(1,2,0).cpu().numpy()

fig, ax = plt.subplots(1, figsize=(8,8))
ax.imshow(img_np)

for box in proposals[0][:500]:
    x1, y1, x2, y2 = box.cpu().numpy()
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                             linewidth=1, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

plt.title("Proposal RPN (Setelah NMS)")
plt.show()
